**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/wei-2023"], check=True)
    os.chdir("/content/encore/entries/wei-2023")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Wei et al. (2023) from dandiset 000021

**Paper.** Wei, Nandi, Jia, Siegle, Denman, Lee, Buchin, Van Geit, Mosher, Olsen & Anastassiou
(2023), "Associations between in vitro, in vivo and in silico cell classes in mouse primary
visual cortex", Nature Communications 14, 2344, doi:10.1038/s41467-023-37844-8. The claims
file `claims/wei-2023.yaml` was written from the version of record (PubMed Central text), its
Supplementary Information and its Source Data workbook.

**Dataset.** DANDI:000021, "Allen Institute - Visual Coding - Neuropixels (Brain Observatory
1.1 Stimulus Set)", version 0.251116.2246 (doi:10.48324/dandi.000021/0.251116.2246). The paper
names the dandiset without a version; 0.251116.2246 is its only published version, so it is
pinned. The dandiset holds 32 sessions of the Brain Observatory 1.1 set; 13 of the paper's 25
wild-type sessions are among them (the other 12 are not, see source-data-units-link), and 5 Pvalb and 6 Sst opto-tagging sessions (the paper
used 8 and 12 mice).

**Inputs from outside the dataset.** The paper's Source Data workbook (from the journal's
site), the code's unit tables and model features (github.com/yinawei/Mouse_V1_EAP_Analysis at
commit 1e35126f1e521b4c98e9ba5fd6a4a8abce70530a), and the in vitro features of the Allen Cell
Types Database API.

| Claim | Tested here |
|---|---|
| source-data-units-link | tested (link, with labels) |
| fig1d-unit-count | tested |
| fig2-one-channel-two-clusters | tested |
| fig2-fs-rs-counts | tested |
| figs3b-fs-faster | tested |
| fig2-rs-more-asymmetric | tested |
| fig2d-three-subclusters-each | tested |
| figs4-rs1-faster-above, figs4-rs-spread, figs4-fs-spread-absent | tested |
| figs5-depth-distribution | tested |
| fig6a-fs1-symmetric | tested |
| fig4g-experiment-classifier-fs-rs, fig5d-experiment-classifier-rs | tested |
| fig3a-isi-fs-rs, fig3a-rs1-excitability, fig3a-rs23-visual-responses, fig3a-fs-visual-absent | tested |
| fig3c-fs-more-phase-locked, fig3c-rs-prefers-theta, figs7-spontaneous-less-locked | tested |
| fig3c-fs-stronger-earlier, fig3d-rs2-gamma-locking, fig3e-fs-cluster-locking | tested |
| figs6c-fs-stronger-across-depth, figs6d-rs3-supragranular-beta, figs6c-granular-phase | tested |
| fig7-opto-unit-counts, fig7d-pvalb-fs-sst-diffuse, fig7e-pvalb-sst-one-channel | tested |
| fig7g-pvalb-more-symmetric, fig7h-pvalb-higher-cv, fig7h-pvalb-theta-sst-later | tested |
| fig4d-ap-sp-models-tpw-rep, fig4f-model-classifier, fig4g-models-map-to-fs-rs, fig4h-cre-lines-map | tested (model features, an input) |
| fig5a-sp-models-three-clusters, fig5c-sp-rs-two-way-mapping, fig6b-pvalb-sst-models-symmetry | tested (model features, an input) |
| fig4e-in-vitro-ap-sp, fig5j-sp1-rate-tau, fig6g-pvalb-sst-in-vitro | tested (Allen Cell Types API and Source Data) |
| fig5e-ca-lva, fig6cf-kv31, fig5g-bifurcation-velocity-sp, fig6d-sst-asymmetric-morphology, fig6e-bifurcation-velocity-inhibitory | tested on the Source Data values |
| figs8-rotation-symmetry | blocked (simulations, models not public) |

Each section prints the results under every reading the criterion names. Where a claim
depends on the cluster labels, it is run on the clusters found here under isolation readings
(c), (a) and (b), and on the Source Data labels of the 566 held units, with this dataset's
measures. Reading (c), reading (a) with at least 100 spikes in the first 5-min gray screen,
is an assumption (see fig1d-unit-count): it keeps 603 of the 603 units of the code's unit
tables that pass reading (a) and adds 27, where reading (a) adds 297. The outcomes of the
claims after fig1d-unit-count are judged on it. Depth relative to layer 4 is taken from the
layer 4 centre computed here from the flash CSD, the method the criterion of
figs5-depth-distribution names, with the layer 4 centre of the code's unit tables (one value
per probe) as a variant (see figs5-depth-distribution).

## Setup and loading

Every session is streamed from the archive and reduced as it loads
(`wei2023.process_session`): every VISp unit of the units table (quality "good" or "noise"), their mean waveforms and features,
spike-train measures, drifting-grating metrics, opto-tagging responses, the CSD layer 4
centre and the spike phases in five LFP bands. Four sessions are processed at a time; each
worker holds about 2 GB at its peak (measured on one session while preparing).

In [ ]:
import os, zlib, time, resource
os.environ.setdefault("OMP_NUM_THREADS", "1")
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import wei2023 as W

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)


def seed_of(claim):
    """The seed of a claim's generator (docs/protocol.md)."""
    return zlib.crc32(f"wei-2023/{claim}".encode())


SEED_PHASE = seed_of("fig3c-fs-more-phase-locked")
SOURCE_DATA = ("https://static-content.springer.com/esm/art%3A10.1038%2Fs41467-023-37844-8/"
               "MediaObjects/41467_2023_37844_MOESM3_ESM.xlsx")
CODE = ("https://raw.githubusercontent.com/yinawei/Mouse_V1_EAP_Analysis/"
        "1e35126f1e521b4c98e9ba5fd6a4a8abce70530a/Examples/data/")
t0 = time.time()
S = W.sessions_to_use()
ALL_SESSIONS = S["wt"] + S["Pvalb"] + S["Sst"]
R = W.process_all(ALL_SESSIONS, SEED_PHASE, workers=4)
units = W.assemble(R, S)
print(f"processed {len(R)} sessions in {time.time() - t0:.0f} s")

In [ ]:
import io, urllib.request


def fetch(url):
    with urllib.request.urlopen(url) as r:
        return io.BytesIO(r.read())


sd_book = pd.ExcelFile(fetch(SOURCE_DATA))
sd = {name: pd.read_excel(sd_book, sheet_name=name, header=None) for name in sd_book.sheet_names}


def sd_table(name, header_row=0):
    d = sd[name]
    t = d.iloc[header_row + 1:].copy()
    t.columns = [str(c) for c in d.iloc[header_row]]
    return t.reset_index(drop=True)


sd_fig2d = sd_table("Fig2d").set_index("unit_id")
sd_fig2d.columns = ["idx", "cluster", "code_label", "TPW", "REP", "inv_below", "inv_above"]
sd_fig2d["onech"] = np.where(sd_fig2d.cluster.str.startswith("FS"), "FS", "RS")
d1 = sd["Fig 1d"]
start = d1.index[d1[1] == "animal_id"][0]
sd_sessions = d1.iloc[start + 1:start + 26, 1:3].astype(int)
sd_sessions.columns = ["session", "units"]
unitinfo = pd.concat([pd.read_csv(fetch(f"{CODE}wt_EAP_data_info/{s}unitinfo.csv"), index_col=0)
                      .assign(session=s) for s in S["wt"]]).set_index("units")
model = pd.read_csv(fetch(CODE + "model_features.csv"), index_col=0)
print("Source Data sheets:", len(sd), "; Fig2d units:", len(sd_fig2d),
      "; code unit tables:", len(unitinfo), "rows; models:", len(model))

## Data check

In [ ]:
info = pd.DataFrame([R[s]["info"] for s in ALL_SESSIONS]).set_index("session")
info["group"] = [g for s in ALL_SESSIONS for g in S if s in S[g]]
info["visp_units"] = units.groupby("session").size()
info["visp_good"] = units[units.quality == "good"].groupby("session").size()
print("Sessions held, by group:", {g: len(v) for g, v in S.items()})
print("Paper's wild-type sessions not in this dandiset:",
      [s for s in W.PAPER_WT_SESSIONS if s not in S["wt"]])
print(info[["group", "genotype", "sex", "visp_units", "visp_good", "visp_noise_units", "n_dg", "dg_time", "dg_block_time",
            "spont_time", "n_flashes", "n_opto_pulses", "t_first", "t_last"]].to_string())
print("\nSex by group:", info.groupby(["group", "sex"]).size().to_dict())
print("\nUnits (VISp, any quality), all sessions:", len(units), "; of quality good:", int((units.quality == "good").sum()))
print("Electrodes tables: CCF x, y, z NaN in", int((info.electrodes_xyz_nan > 0).sum()), "of", len(info), "sessions;",
      "sessions with coordinates:", info.index[info.electrodes_xyz_nan == 0].tolist(),
      "; channels with y equal to z there:", info.loc[info.electrodes_xyz_nan == 0, "electrodes_y_eq_z"].tolist(),
      "of", info.loc[info.electrodes_xyz_nan == 0, "electrodes_n"].tolist())
nan_ses = info[info.electrodes_xyz_nan > 0]
print("Sessions with NaN coordinates: channels", nan_ses.electrodes_n.to_dict(), "; NaN values over x, y and z",
      nan_ses.electrodes_xyz_nan.to_dict(), "; x, y and z NaN on every channel:",
      bool((nan_ses.electrodes_xyz_nan == 3 * nan_ses.electrodes_n).all()))
for c, unit in [("A", "uV (one-channel EAP amplitude, file says volts)"), ("TPW", "ms"), ("REP", "ms"),
                ("PL_below", "ms/mm"), ("PL_above", "ms/mm"), ("rate_session", "Hz"),
                ("rate_spont5", "Hz"), ("rate_dg", "Hz"), ("depth_csd", "um")]:
    v = units[c].astype(float)
    print(f"{c:13s} {unit:50s} range {np.nanmin(v):9.3f} to {np.nanmax(v):9.3f}, median {np.nanmedian(v):8.3f}, NaN {v.isna().sum()}")
print("\nWindow of 40 channels does not fit on the probe:", int((units.fits == False).sum()), "units")
print("Opto pulse counts (10 ms square):", info.loc[S["Pvalb"] + S["Sst"], "n_opto_pulses"].to_dict())

In [ ]:
# Checks of the description against the data.
# (1) Waveform scale: amplitudes are tens to hundreds, so the waveforms are in microvolts.
# (2) The code's per-unit firing rate column fr (unit tables of the code) against the units
#     table's firing_rate, and fr_5min against the rate in the first 5-min gray screen.
j = units.join(unitinfo[["fr", "fr_5min", "depth", "good"]], how="inner")
print("code unit-table rows matched to VISp good units here:", len(j), "of", len(unitinfo))
print("max |firing_rate - code fr|:", float(np.max(np.abs(j.firing_rate - j.fr))))
for c in ["rate_spont5", "rate_spont", "rate_first5min", "rate_session"]:
    print(f"median |{c} - code fr_5min| = {np.median(np.abs(j[c] - j.fr_5min)):.4f} Hz")
# (3) Stimulus clock: spikes of VISp units respond to the flashes and gratings at their onsets
#     (rate 30-80 ms after onset against 30 ms before, pooled over units), and the contrast
#     vanishes when the onsets are shifted by half a trial.
def onset_contrast(session, table, shift=0.0):
    f = W.open_nwb(W.session_paths()[session])
    tab = W._table(f["intervals"], table)
    on = tab.start_time.values + shift
    u = units[units.session == session]
    sp = W.read_spikes(f, u.row.values)
    post = sum(W.count_within(np.sort(x), on + 0.03, on + 0.08) for x in sp.values())
    pre = sum(W.count_within(np.sort(x), on - 0.03, on) for x in sp.values())
    f.close()
    return (post / 0.05) / max(pre / 0.03, 1e-9)
for s in [S["wt"][0], S["Pvalb"][0], S["Sst"][0]]:
    print(s, "flash onset ratio post/pre: true %.2f, shifted 0.125 s %.2f" % (
        onset_contrast(s, "flashes_presentations"), onset_contrast(s, "flashes_presentations", 0.125)))

## source-data-units-link

> The 1204 unit ids of the paper's Source Data, with their one- and multi-channel cluster
> labels, are units of the dataset's wild-type V1 recordings.

Each unit id of Source Data sheet Fig2d is looked up among the units of the sessions held.
Unit ids are unique across the Allen sessions, so a unit found in a session's table belongs
to that session. Ids not among the VISp units of quality "good" are looked up in the full
units tables of the held sessions, to say where they are.

In [ ]:
held = set(S["wt"])
sd_counts = sd_sessions.set_index("session").units
print("Source Data units in the 13 held sessions (sheet Fig 1d):", int(sd_counts.loc[S["wt"]].sum()),
      "; in the 12 missing sessions:", int(sd_counts.drop(S["wt"]).sum()))
wt_units = units[units.group == "wt"]
found = sd_fig2d.index.intersection(wt_units.index)
per = wt_units.loc[found].groupby("session").size().reindex(S["wt"]).fillna(0).astype(int)
cmp = pd.DataFrame({"source_data": sd_counts.loc[S["wt"]], "found_visp_good": per})
print(cmp.to_string())
print("found:", len(found), "of", int(sd_counts.loc[S["wt"]].sum()))
# ids not found among VISp good units: where are they?
rest = sd_fig2d.index.difference(found)
# every session file of the dandiset, not only the 13 wild-type sessions held
all_paths = W.session_paths()
missing = [s for s in W.PAPER_WT_SESSIONS if s not in S["wt"]]
print("session files in the dandiset:", len(all_paths), "; of the 12 missing session ids among them:",
      [s for s in missing if s in all_paths])
elsewhere = {}
for s, path in sorted(all_paths.items()):
    f = W.open_nwb(path)
    ids = f["units/id"][:]
    hit = np.isin(rest, ids)
    if hit.any():
        elsewhere[s] = int(hit.sum())
    f.close()
print("Source Data ids found outside the VISp good units of the 13 sessions, in any of the", len(all_paths),
      "session files:", elsewhere)
print("Source Data ids in no session file of the dandiset:", len(rest) - sum(elsewhere.values()),
      "(sheet Fig 1d puts", int(sd_counts.drop(S["wt"]).sum()), "units in the missing sessions)")
# do the matched units agree on what both describe? features of the Source Data and here
m = wt_units.loc[found]
for a, b in [("TPW", "TPW"), ("REP", "REP"), ("PL_below", "inv_below"), ("PL_above", "inv_above")]:
    d = np.abs(m[a].astype(float) - sd_fig2d.loc[found, b].astype(float))
    print(f"{a:9s} |here - Source Data|: equal (<1e-6) for {np.mean(d < 1e-6):.1%}, within 5% of the "
          f"Source Data range for {np.mean(d < 0.05 * np.ptp(sd_fig2d[b].astype(float))):.1%}, median {np.nanmedian(d):.2e}")
# where the features differ: per session and feature, the share of matched units equal to the Source Data
eq = pd.DataFrame({a: np.abs(m[a].astype(float) - sd_fig2d.loc[found, b].astype(float)) < 1e-6
                   for a, b in [("TPW", "TPW"), ("REP", "REP"), ("PL_below", "inv_below"), ("PL_above", "inv_above")]})
eq["session"] = m.session.values
print("share of matched units whose feature equals the Source Data, by session (%):")
print((100 * eq.groupby("session").mean()).round(1).assign(units=eq.groupby("session").size()).to_string())
both = eq.TPW & eq.REP
print("units with TPW and REP equal:", int(both.sum()), "; with TPW or REP different:", int((~both).sum()),
      "; of those, 1/V below also different:", int((~both & ~eq.PL_below).sum()))
dtp = (m.TPW.astype(float) - sd_fig2d.loc[found, "TPW"].astype(float))[~both.values]
print("TPW here minus Source Data where they differ: median %.3f ms, IQR %.3f to %.3f ms" % (
      dtp.median(), dtp.quantile(0.25), dtp.quantile(0.75)))
print("matched units passing isolation reading (a):", int(m.iso_a.sum()), "of", len(m))
print("code unit tables (good flag) against reading (a), units of the held sessions:")
print(pd.crosstab(units.loc[unitinfo.index, "iso_a"], unitinfo.good, rownames=["reading (a)"], colnames=["code good"]))
# which reading (a) units are not in the code's tables? spikes in the first 5-min gray screen
a_ = wt_units[wt_units.iso_a].assign(in_code=lambda d: d.index.isin(unitinfo.index))
print("reading (a) units in the code's tables against at least 100 spikes in the first 5-min gray screen:")
print(pd.crosstab(a_.in_code, a_.n_spont5 >= 100, rownames=["in code tables"], colnames=[">= 100 spikes"]))
print("medians by membership:", a_.groupby("in_code")[["n_spont5", "rate_spont5", "firing_rate", "snr", "isi_violations"]]
      .median().round(3).to_dict("index"))
print("code tables: fr_5min minimum %.3f Hz; first 5-min gray screen lasts %.0f to %.0f s" % (
      unitinfo.fr_5min.min(), min(R[s]["info"]["spont5"][1] - R[s]["info"]["spont5"][0] for s in S["wt"]),
      max(R[s]["info"]["spont5"][1] - R[s]["info"]["spont5"][0] for s in S["wt"])))
print("Source Data units with fewer than 100 spikes there:", int((m.n_spont5 < 100).sum()), "of", len(m),
      "; reading (b) units of quality noise among the wild-type units:", int((wt_units.quality != "good").sum()))

**Result: partially reproduced.** The dandiset holds 13 of the 25 sessions; all 566 Source Data units of those sessions are VISp units of quality good there, with the per-session counts of sheet Fig 1d, and none of the 638 units of the other 12 sessions is in any of the dandiset's 32 session files, which do not include those sessions. On the matched units TPW and REP are equal to the Source Data values for 85.0% of units and 1/V below and above for 71.7% and 68.0%. The 85 units whose TPW or REP differ also differ in 1/V below, and the per-session table shows where they lie; where they differ, TPW differs by a median of 0.004 ms (IQR -0.017 to 0.024 ms), so the waveforms are close but not identical. Every one of the 566 units has at least 100 spikes in the first 5-min gray screen, and 270 of the 297 reading (a) units outside the code's tables have fewer. The other claims are judged on the 13 sessions.

## fig1d-unit-count

> The analysis rests on 1204 well-isolated V1 units from 25 wild-type mice, an average of 48
> units per mouse.

The dandiset holds 13 of the 25 sessions, so the total of 1204 cannot be reached; following the
link claim, the counts are judged on the held subset, where the Source Data put 566 units
(43.5 per mouse), and the mean per mouse is compared with 48. Readings (a) the AllenSDK default
filter on units of quality "good" and (b) every VISp unit the units table lists (quality "good"
or "noise"), each followed by the code's exclusions. A third reading, (c), is not one the
criterion names: reading (a) with at least 100 spikes in the first 5-min gray screen. It is
taken from the Results ("well-isolated ... during spontaneous activity", paragraph 1 of "EAP
recordings from in vivo experiments") and the Methods' 100-spike rule ("Single unit firing
properties"), and from the data check above: every unit of the code's tables has at least 100
spikes there, while 270 of the 297 reading (a) units outside them have fewer.
A variant starts from the code's own unit tables (their good flag), which the paper's code
prepared from the NWB files.

In [ ]:
sel = {r: W.select_wt(units, r) for r in "abc"}
rows = []
for r in "abc":
    w = sel[r]
    n_iso = int((units.group.eq("wt") & units[f"iso_{r}"]).sum())
    per = w.groupby("session").size().reindex(S["wt"]).fillna(0)
    rows.append({"reading": r, "isolated": n_iso, "after exclusions": len(w), "sessions": len(S["wt"]),
                 "mean per mouse": per.mean(), "min": per.min(), "max": per.max()})
# variant: the code's unit tables
v = units[(units.group == "wt") & units.index.isin(unitinfo.index[unitinfo.good == 1]) & ~units.bad_list]
v = v[W.code_exclusions(v).values]
per_v = v.groupby("session").size().reindex(S["wt"]).fillna(0)
rows.append({"reading": "code tables", "isolated": int((unitinfo.good == 1).sum()), "after exclusions": len(v),
             "sessions": len(S["wt"]), "mean per mouse": per_v.mean(), "min": per_v.min(), "max": per_v.max()})
lit = {r: W.select_wt(units, r, nan_aware=False) for r in "abc"}
for r in "abc":
    rows.append({"reading": f"{r}, code's exclusion literally", "isolated": np.nan, "after exclusions": len(lit[r]),
                 "sessions": 13, "mean per mouse": len(lit[r]) / 13, "min": np.nan, "max": np.nan})
print(pd.DataFrame(rows).to_string(index=False))
print("NaN in the features of reading (a) before exclusions:",
      units[(units.group == "wt") & units.iso_a][["PL_above", "W_above", "PL_below", "W_below"]].isna().sum().to_dict())
print("Source Data, same 13 sessions: 566 units, mean per mouse", round(566 / 13, 1),
      "; expected 48 +- 4.8 per mouse")
print("\nper session:")
print(pd.DataFrame({"a": sel["a"].groupby("session").size(), "b": sel["b"].groupby("session").size(),
                    "c": sel["c"].groupby("session").size(),
                    "code tables": per_v, "Source Data": sd_counts.loc[S["wt"]]}).to_string())
for r in "abc":
    print(f"reading ({r}): {len(sel[r].index.intersection(sd_fig2d.index))} of the 566 Source Data units kept, "
          f"{len(sel[r].index.difference(sd_fig2d.index))} units the Source Data do not hold")
print("\noverlap of the code-table variant with the Source Data units:",
      len(v.index.intersection(sd_fig2d.index)), "of", len(v), "here and", 566, "there")

**Result: not reproduced.** On the 13 held sessions, reading (a) gives 741 units after the code's exclusions, 57.0 per mouse, and reading (b) 1820, 140.0 per mouse, against 48 +- 4.8; the Source Data put 566 units (43.5 per mouse) in the same sessions. Reading (c), not a reading of the criterion, gives 516 units, 39.7 per mouse, and keeps 487 of the 566 Source Data units with 29 others, where reading (a) keeps 493 with 248 others. Starting from the code's own unit tables gives 494 units, 38.0 per mouse. The total of 1204 cannot be reached with 13 of 25 sessions.

## fig2-one-channel-two-clusters

> K-means clustering of the one-channel features TPW and REP gives two clusters as the optimum
> by both the elbow method and the density function, a narrow-waveform cluster with short REP
> and a wide-waveform cluster with long REP.

The code's procedure (NP_EAP_FeatureAnalysis.ipynb cells 11 to 13): median imputation,
z-score, K-means with k-means++ and 1000 initialisations for K = 1 to 9, the elbow by
ecp_func.findElbow and the minimum of f(K). The random state of K-means is the claim's seed
(the code uses 10 and 11400); the four other seeds are shown. Run on both isolation readings.

In [ ]:
SEED_C1 = seed_of("fig2-one-channel-two-clusters")
one = {}
for r in "cab":
    w = sel[r]
    one[r] = W.cluster_one_channel(w, SEED_C1)
    res = one[r]
    fsm = res["fsrs"] == "FS"
    print(f"reading ({r}), {len(w)} units: elbow K = {res['elbow_k']}, f(K) minimum at K = {res['fk_k']}")
    print("   WCSS/S1:", np.round(res["wcss"] / res["wcss"][0], 3))
    print("   f(K):   ", np.round(res["fk"], 3))
    print(f"   FS (lower TPW) median TPW {w.TPW[fsm].median():.3f} ms, REP {w.REP[fsm].median():.3f} ms; "
          f"RS median TPW {w.TPW[~fsm].median():.3f} ms, REP {w.REP[~fsm].median():.3f} ms")
seeds5 = [SEED_C1 + k * 1000 for k in range(5)]
for r in "cab":
    ks = []
    for s_ in seeds5[1:]:
        if r == "c":
            Xs = W.pre_process(sel[r][["TPW", "REP"]].values)
            wc, fk = W.selection_k(Xs, 9, seed=s_)
            ks.append((W.find_elbow(wc), int(fk.argmin() + 1)))
    if ks:
        print(f"reading ({r}), (elbow K, f(K) K) under seeds +1000..+4000:", ks)

**Result: reproduced.** Under all three readings the elbow and the minimum of f(K) select K = 2 (also under the four other seeds, reading c), and the cluster of lower median TPW (0.258 ms, reading c) also has the lower median REP (0.250 against 0.303 ms). The Source Data features of the 1204 units give K = 2 by both methods as well.

## fig2-fs-rs-counts

> Of the 1204 units, 281 (23.3%) fall in the narrow-waveform (FS) cluster and 923 (76.7%) in
> the wide-waveform (RS) cluster.

The unit set differs from the paper's (13 of 25 sessions), so the fraction is the measure
(criterion). Agreement with the Source Data labels on the shared units is shown.

In [ ]:
from sklearn.cluster import KMeans
for r in "cab":
    w = sel[r]
    lab = pd.Series(one[r]["fsrs"], index=w.index)
    n_fs, n_rs = int((lab == "FS").sum()), int((lab == "RS").sum())
    shared = lab.index.intersection(sd_fig2d.index)
    agree = np.mean(lab.loc[shared].values == sd_fig2d.loc[shared, "onech"].values)
    print(f"reading ({r}): FS {n_fs}, RS {n_rs}, FS fraction {100 * n_fs / len(w):.1f}% "
          f"(expected 23.3 +- 2.33); shared with Source Data {len(shared)}, labels agree {agree:.1%}")
    print("   per session FS fraction:", (lab.eq("FS").groupby(w.session).mean() * 100).round(1).to_dict())
    X = W.pre_process(w[["TPW", "REP"]].values)
    fr = []
    for s_ in seeds5:
        l = KMeans(2, init="k-means++", n_init=1000, random_state=s_).fit(X).labels_
        fsl = 0 if np.median(w.TPW.values[l == 0]) < np.median(w.TPW.values[l == 1]) else 1
        fr.append(round(100 * np.mean(l == fsl), 2))
    print("   FS fraction under the five seeds:", fr)
sdl = sd_fig2d.loc[sd_fig2d.index.intersection(units.index)]
print("Source Data labels on the 566 held units: FS", int((sdl.onech == "FS").sum()),
      f"({100 * (sdl.onech == 'FS').mean():.1f}%)")

**Result: reproduced, resting on an assumption.** Under reading (c) the FS fraction is 25.4% (131 of 516), within 23.3 +- 2.33%, under all five seeds; the FS/RS labels agree with the Source Data on 99.4% of shared units. Readings (a) and (b) give 20.8% (154 of 741) and 20.5% (373 of 1820), below the tolerance, and the Source Data labels of the 566 held units give 24.0%. The fraction depends on the unit set: the low-firing units of reading (a) that the code's tables leave out lower it.

## figs3b-fs-faster

> FS units fire faster than RS units, at a median of 4.85 Hz (IQR 1.93 to 10.79 Hz) against
> 2.05 Hz (IQR 0.84 to 5.00 Hz).

Rate as N / T. Readings of the period (criterion): (a) the spontaneous (gray screen) block,
shown as (a1) the first 5-min gray-screen block, which reproduces the code's column fr_5min
exactly (data check above), and (a2) all gray-screen intervals of 30 s or more; (b) the first
5 minutes of the recording (from the first stimulus presentation); (c) the whole session.

In [ ]:
def q(v):
    return f"{np.median(v):.2f} (IQR {np.percentile(v, 25):.2f} to {np.percentile(v, 75):.2f})"
for r in "cab":
    w = sel[r]
    lab = one[r]["fsrs"]
    for col, name in [("rate_spont5", "a1 first 5-min gray screen"), ("rate_spont", "a2 all gray screen"),
                      ("rate_first5min", "b first 5 min"), ("rate_session", "c whole session")]:
        fs_, rs_ = w[col][lab == "FS"], w[col][lab == "RS"]
        p = stats.mannwhitneyu(fs_, rs_, alternative="two-sided").pvalue
        print(f"({r}) {name:28s} FS {q(fs_)} Hz, RS {q(rs_)} Hz, Mann-Whitney p = {p:.2e}")
sdl = sd_fig2d.loc[sd_fig2d.index.intersection(units.index)]
uu = units.loc[sdl.index]
fs_, rs_ = uu.rate_spont5[sdl.onech == "FS"], uu.rate_spont5[sdl.onech == "RS"]
print(f"variant, Source Data units and labels (566), a1: FS {q(fs_)}, RS {q(rs_)}, "
      f"p = {stats.mannwhitneyu(fs_, rs_, alternative='two-sided').pvalue:.2e}")

**Result: not reproduced (the criterion defines only a reproduced outcome and no reading meets it; changed from partially reproduced after round 3 of verification, which agreed).** FS units fire faster than RS units under every period and isolation reading (Mann-Whitney p from 1.3e-05 to 2.3e-64), but no reading puts both medians within tolerance. Under reading (c) the first 5-min gray screen (the period of the code's fr_5min column) gives FS 4.26 and RS 2.29 Hz, the first 5 minutes FS 4.50 (within tolerance) and RS 2.38 Hz, against 4.85 and 2.05 Hz. On the Source Data units and labels the gray-screen block gives 4.55 and 2.50 Hz.

## fig2-rs-more-asymmetric

> RS units have more asymmetric EAP propagation below against above the soma than FS units.

Symmetry index (Methods eq. 1) per unit; the paper's two-sample procedure
(ecp_func.two_sample_test).

In [ ]:
for r in "cab":
    w = sel[r]
    lab = one[r]["fsrs"]
    si = W.symmetry_index(w.PL_below, w.PL_above)
    test, p = W.two_sample_test(si[lab == "RS"], si[lab == "FS"])
    print(f"({r}) symmetry index median RS {np.nanmedian(si[lab == 'RS']):.3f}, FS {np.nanmedian(si[lab == 'FS']):.3f}; "
          f"{test} p = {p:.2e}; NaN {int(np.isnan(si).sum())} units")

**Result: reproduced.** The median symmetry index is 1.599 for RS and 0.784 for FS (reading c; Mann-Whitney p = 2.3e-21), and RS is higher under readings (a) and (b) as well (p = 3.6e-26 and 2.6e-53).

## fig2d-three-subclusters-each

> Clustering FS and RS separately on the multi-channel features 1/V below and 1/V above gives
> three clusters in each, FS1 to FS3 (130, 82 and 69 units) and RS1 to RS3 (479, 235 and 209
> units).

As the code (cells 17 to 21): 1/V below and above median-imputed and z-scored over all units,
K-means within FS and within RS with K = argmin f(K) + 1; the elbow is also reported. Clusters
are named by their greatest overlap with the Source Data labels on shared units. As the unit
set differs, the counts are also given as fractions of FS and RS (the paper's 46.3, 29.2 and
24.6% of FS; 51.9, 25.5 and 22.6% of RS).

In [ ]:
SEED_C6 = seed_of("fig2d-three-subclusters-each")
multi, six = {}, {}
for r in "cab":
    w = sel[r]
    multi[r] = W.cluster_multi_channel(w, one[r]["fsrs"], SEED_C6)
    lab6 = pd.Series("", index=w.index)
    for grp in ("FS", "RS"):
        m = multi[r][grp]
        names, tab = W.name_by_overlap(m["index"], m["labels"], sd_fig2d.cluster, [f"{grp}{i}" for i in (1, 2, 3, 4, 5, 6, 7, 8, 9)])
        lab6.loc[m["index"]] = names
        print(f"({r}) {grp}: f(K) minimum at K = {m['fk_k']}, elbow K = {m['elbow_k']}; f(K) = {np.round(m['fk'], 3)}")
        print("   overlap with Source Data labels (rows: clusters here):")
        print("   " + tab.to_string().replace("\n", "\n   "))
    six[r] = lab6
    cnt = lab6.value_counts().sort_index()
    print(f"({r}) counts:", cnt.to_dict())
    print(f"({r}) fractions within FS and RS (%):",
          {k: round(100 * v / (lab6.str[:2] == k[:2]).sum(), 1) for k, v in cnt.items()})
    shared = lab6.index.intersection(sd_fig2d.index)
    print(f"({r}) six-cluster labels agree with the Source Data on {np.mean(lab6.loc[shared] == sd_fig2d.loc[shared, 'cluster']):.1%} of {len(shared)} shared units")
# seeds: K-means at the chosen K under the five seeds, reading (c)
w = sel["c"]
X = W.pre_process(w[["PL_below", "PL_above"]].values)
for grp in ("FS", "RS"):
    m = one["c"]["fsrs"] == grp
    sizes = []
    for s_ in [SEED_C6 + k * 1000 for k in range(5)]:
        l = KMeans(multi["c"][grp]["k"], init="k-means++", n_init=1000, random_state=s_).fit(X[m]).labels_
        sizes.append(sorted(np.bincount(l).tolist(), reverse=True))
    print(f"(c) {grp} cluster sizes under the five seeds:", sizes)
# the code's exclusion run literally (NaN turns off the 3 SD exclusion of 1/V above)
for r in "ca":
    wl = lit[r]
    ol = W.cluster_one_channel(wl, SEED_C1)
    ml = W.cluster_multi_channel(wl, ol["fsrs"], SEED_C6)
    print(f"({r}) literal exclusions, {len(wl)} units: FS K = {ml['FS']['k']} sizes {sorted(np.bincount(ml['FS']['labels']).tolist(), reverse=True)}, "
          f"RS K = {ml['RS']['k']} sizes {sorted(np.bincount(ml['RS']['labels']).tolist(), reverse=True)}")
# with K forced to 3 when f(K) picks another K
for r in "cab":
    for grp in ("FS", "RS"):
        if multi[r][grp]["k"] != 3:
            m3 = W.cluster_multi_channel(sel[r], one[r]["fsrs"], SEED_C6, k_override=3)[grp]
            names, _ = W.name_by_overlap(m3["index"], m3["labels"], sd_fig2d.cluster, [f"{grp}{i}" for i in (1, 2, 3)])
            print(f"({r}) {grp} with K forced to 3 (variant):", pd.Series(names).value_counts().sort_index().to_dict())

In [ ]:
# The paper's own features: the 1204 units of Source Data sheet Fig2d, clustered by the same
# procedure (a check of the clustering apart from the feature differences of this dataset).
X2 = W.pre_process(sd_fig2d[["TPW", "REP"]].astype(float).values)
wc, fk = W.selection_k(X2, 9, seed=SEED_C1)
print(f"Source Data features, one-channel, 1204 units: elbow K = {W.find_elbow(wc)}, f(K) minimum at K = {int(fk.argmin() + 1)}")
X6 = W.pre_process(sd_fig2d[["inv_below", "inv_above"]].astype(float).values)
for grp in ("FS", "RS"):
    m = (sd_fig2d.onech == grp).values
    wc, fk = W.selection_k(X6[m], 9, seed=SEED_C6)
    l3 = W.kmeans_labels(X6[m], 3, SEED_C6)
    names, tab = W.name_by_overlap(sd_fig2d.index[m], l3, sd_fig2d.cluster, [f"{grp}{i}" for i in (1, 2, 3)])
    print(f"Source Data features, {grp} ({m.sum()} units): f(K) minimum at K = {int(fk.argmin() + 1)}, elbow K = {W.find_elbow(wc)}; "
          f"f(K) = {np.round(fk, 3)}")
    print(f"   K = 3 counts {pd.Series(names).value_counts().sort_index().to_dict()}, agreement with the paper's labels "
          f"{np.mean(names == sd_fig2d.cluster.values[m]):.1%}")


The clusters used by the claims that depend on this one. RS takes the K that f(K) selects (3
under readings c and a; under reading b f(K) selects 1, and K is forced to 3). For FS, f(K)
selects 2 (readings c and a) or 9 (reading b), so no FS3 exists; FS is clustered with K forced
to 3 so that the FS1 to FS3 claims can be tested (a deviation), and each such claim is also
run on the Source Data labels of the 566 held units.

In [ ]:
labels = {}
for r in "cab":
    w = sel[r]
    lab6 = pd.Series("", index=w.index)
    for grp in ("FS", "RS"):
        m = multi[r][grp] if multi[r][grp]["k"] == 3 else \
            W.cluster_multi_channel(w, one[r]["fsrs"], SEED_C6, k_override=3)[grp]
        names, _ = W.name_by_overlap(m["index"], m["labels"], sd_fig2d.cluster, [f"{grp}{i}" for i in (1, 2, 3)])
        lab6.loc[m["index"]] = names
    labels[r] = pd.DataFrame({"onech": one[r]["fsrs"], "six": lab6.values}, index=w.index)
    print(f"({r}) clusters used downstream:", labels[r].six.value_counts().sort_index().to_dict())
held_sd = sd_fig2d.loc[sd_fig2d.index.intersection(units.index)]
labels["sd"] = pd.DataFrame({"onech": held_sd.onech, "six": held_sd.cluster})
print("(sd) Source Data labels of the held units:", labels["sd"].six.value_counts().sort_index().to_dict())
READINGS = {"c": "reading (c)", "a": "reading (a)", "b": "reading (b)", "sd": "Source Data labels"}

**Result: not reproduced.** RS gives K = 3 by both methods under readings (c) and (a) (180, 95 and 110 units, reading c; 46.8, 24.7 and 28.6% of RS against the paper's 51.9, 25.5 and 22.6%), but FS gives K = 2 by both methods under readings (c) and (a), and under reading (b) f(K) selects 9 for FS and 1 for RS. The six-cluster labels agree with the Source Data on 80.7% of shared units (reading c). With K forced to 3, FS gives 59, 41 and 31 units (reading c). The same procedure on the paper's own features (Source Data sheet Fig2d) selects K = 3 for FS and RS by both methods and returns the paper's labels for all 1204 units, so the difference comes from the features recomputed here (see source-data-units-link), not from the clustering. With the code's exclusions run literally, a single outlier unit forms one of the three RS clusters (readings c and a).

## figs4-rs1-faster-above

> RS1 and RS2 both propagate asymmetrically and differ mainly in supragranular propagation
> velocity, faster in RS1 than in RS2.

Kruskal-Wallis with Holm-corrected Mann-Whitney tests on 1/V above (Fig S4 legend). In the
Source Data variant the features are this dataset's.

In [ ]:
def three(df, col, groups):
    return W.kruskal_holm({g: df.loc[df.six == g, col].astype(float).values for g in groups})


def table_for(r):
    lab = labels[r]
    return units.loc[lab.index].assign(onech=lab.onech, six=lab.six,
                                       spread=lambda d: d.spread_below + d.spread_above,
                                       si=lambda d: W.symmetry_index(d.PL_below, d.PL_above))


T = {r: table_for(r) for r in READINGS}
for r in READINGS:
    for col in ["PL_above", "PL_below"]:
        k = three(T[r], col, ["RS1", "RS2", "RS3"])
        print(f"{READINGS[r]:18s} {col}: medians {dict((g, round(v, 3)) for g, v in k['medians'].items())}, "
              f"KW p = {k['p']:.2e}, Holm RS1-RS2 p = {k['post']['RS1-RS2']:.2e}, n {k['n']}")

**Result: reproduced.** The median 1/V above is 1.468 ms/mm in RS1 and 3.259 in RS2 (reading c; Kruskal-Wallis p = 9.0e-47, Holm RS1 against RS2 p = 2.8e-41), so RS1 propagates faster above the soma; readings (a), (b) and the Source Data labels give the same.

## figs4-rs-spread

> RS1 to RS3 differ in spatial spread, RS3 being the most spatially confined.

In [ ]:
for r in READINGS:
    k = three(T[r], "spread", ["RS1", "RS2", "RS3"])
    print(f"{READINGS[r]:18s} spread medians {dict((g, round(v, 1)) for g, v in k['medians'].items())} um, "
          f"KW H = {k['H']:.1f}, p = {k['p']:.2e}; Holm {dict((g, float('%.2e' % v)) for g, v in k['post'].items())}")

**Result: reproduced.** The spreads differ (Kruskal-Wallis H = 23.8, p = 6.8e-06, reading c) and RS3 is the most confined (median 144.8 um against 171.3 and 150.8); readings (a), (b) and the Source Data labels give the same order with p = 6.4e-07, 1.9e-18 and 4.1e-05. RS3's 1/V below has the most negative median (-0.618, reading c), reported, not judged.

## figs4-fs-spread-absent

> FS1 to FS3 show no significant difference in spatial spread.

In [ ]:
for r in READINGS:
    k = three(T[r], "spread", ["FS1", "FS2", "FS3"])
    print(f"{READINGS[r]:18s} spread medians {dict((g, round(v, 1)) for g, v in k['medians'].items())} um, "
          f"KW H = {k['H']:.2f}, p = {k['p']:.3f}, n {k['n']}")

**Result: reproduced, resting on an assumption.** On the FS clusters of reading (c) (K forced to 3) the Kruskal-Wallis test gives p = 0.399 (medians 155.5, 168.4 and 166.2 um), reading (a) p = 0.978 and the Source Data labels p = 0.267; under reading (b) p is below 0.001.

## figs5-depth-distribution

> The six multi-channel clusters are distributed differently along the V1 depth axis.

Depth relative to the layer 4 centre. The criterion names the method: the CSD of the
flash-averaged LFP, with the earliest sink in V1 as the centre. The centre computed here that
way (the first VISp channel whose CSD reaches half the minimum within 0 to 150 ms) gives
depth_csd, which decides the outcome. The code's unit tables give each unit's depth as its
probe position minus one value per probe; that value gives the depth of every unit on the
probe (depth_code), shown as a variant. The code does not say how it was placed, and the table
below shows how often the same value recurs across probes. Neither centre is validated here;
the CSD of every probe is plotted.

In [ ]:
csd = pd.DataFrame([dict(c, session=s) for s in S["wt"] for c in R[s]["csd"]])
print("CSD centres per probe:", csd[["session", "probe", "centre", "t_first_ms", "n_v1_lfp"]].to_string(index=False))
jj = units.join(unitinfo[["depth"]], how="inner")
dd = jj.depth_csd - jj.depth
print(f"depth here minus the code's depth, {len(jj)} units: median {dd.median():.0f} um, "
      f"within 40 um for {np.mean(np.abs(dd) <= 40):.1%}, within 100 um for {np.mean(np.abs(dd) <= 100):.1%}")
print("per session median difference:", dd.groupby(jj.session).median().to_dict())
# The code's depth is the probe position minus one value per probe (its layer 4 centre); that
# centre gives a depth for every unit of the probe (depth_code), shown as a variant.
jj["offset"] = jj.vpos - jj.depth
off = jj.groupby(["session", "probe_id"]).offset.agg(["median", "min", "max", "size"])
print("code's depth = probe position - one value per probe:", bool((off["min"] == off["max"]).all()))
cmpc = off["median"].rename("code centre").reset_index().merge(
    csd.rename(columns={"probe": "probe_id", "centre": "CSD centre here"})[["session", "probe_id", "CSD centre here", "centre_min", "t_first_ms"]],
    on=["session", "probe_id"])
cmpc["here - code"] = cmpc["CSD centre here"] - cmpc["code centre"]
print(cmpc.to_string(index=False))
print("probes per value of the code's centre:", cmpc["code centre"].value_counts().to_dict(), "of", len(cmpc), "probes")
cen = off["median"].droplevel(0)
units["depth_code"] = units.vpos - units.probe_id.map(cen)
for r in READINGS:
    T[r]["depth_code"] = units.loc[T[r].index, "depth_code"]
print("units without a code centre (a probe the code's tables do not cover), reading (c):", int(T["c"].depth_code.isna().sum()), "of", len(T["c"]))
print("depth_code range (wild-type units of reading c): %.0f to %.0f um" % (T["c"].depth_code.min(), T["c"].depth_code.max()),
      "; depth_csd range: %.0f to %.0f um" % (T["c"].depth_csd.min(), T["c"].depth_csd.max()))
fig, axs = plt.subplots(2, 7, figsize=(17, 6), sharey=False)
for ax, (_, row) in zip(axs.flat, csd.iterrows()):
    zc = row["z"]
    ax.imshow(row["csd"].T, aspect="auto", origin="lower", cmap="RdBu", extent=[row["t_ms"][0], row["t_ms"][-1], zc[0], zc[-1]],
              vmin=-np.abs(row["csd"]).max(), vmax=np.abs(row["csd"]).max())
    ax.axhline(row["centre"], color="k", lw=1)
    ax.axhline(cen.get(row["probe"], np.nan), color="g", lw=1, ls="--")
    ax.set_title(f"{row['session']}", fontsize=8)
    ax.set_xlabel("ms after flash", fontsize=7)
axs.flat[0].set_ylabel("probe position (um)")
axs.flat[-1].axis("off")
fig.suptitle("Flash CSD of the VISp LFP channels (sinks blue); black: centre here, green dashed: the code's centre")
plt.tight_layout()
plt.show()
print("CSD panels: 0 to 150 ms after flash onset, the VISp channels of each probe; colour scale per panel symmetric about 0")
six_names = ["FS1", "FS2", "FS3", "RS1", "RS2", "RS3"]
for r in READINGS:
    g = {c: T[r].loc[T[r].six == c, "depth_csd"].astype(float).values for c in six_names}
    H, p = stats.kruskal(*g.values())
    print(f"{READINGS[r]:18s} depth medians {dict((c, float(np.median(v))) for c, v in g.items())}, KW H = {H:.1f}, p = {p:.2e}")
    gg = {c: T[r].loc[T[r].six == c, "depth_code"].dropna().astype(float).values for c in six_names}
    print(f"   with the code's centre (depth_code, n {sum(len(v) for v in gg.values())}): medians {dict((c, float(np.median(v))) for c, v in gg.items())}, "
          "KW p = %.2e" % stats.kruskal(*gg.values()).pvalue)
    if r == "sd":
        gg = {c: jj.loc[jj.index.intersection(T[r].index[T[r].six == c]), "depth"].values for c in six_names}
        print("   with the code's depth on the units of its tables:", "KW p = %.2e" % stats.kruskal(*gg.values()).pvalue)

**Result: not reproduced, resting on an assumption (the verified outcome; a change to partially reproduced was proposed, and round 3 of verification did not agree with it and would give reproduced, resting on an assumption).** With the CSD layer 4 centre computed here, the method the criterion names, the six clusters differ in depth (Kruskal-Wallis p = 0.044, reading c; 0.011, reading a; 2.0e-08, reading b), which meets the criterion, but on the Source Data labels the same depth gives p = 0.36. With the layer 4 centre of the code's unit tables the test gives p = 0.25 (reading c), 0.29 (reading a), 2.8e-05 (reading b) and 0.34 on the Source Data labels. That centre is 2740 um on 8 of the 13 probes, so it may be a fixed offset and not a placement from each probe's CSD; the recorded outcome took it as the paper's CSD placement, which these outputs do not support. The two centres differ by -420 to +340 um per probe, and neither is validated here. On the criterion's method the difference is present for the clusters found here and absent for the paper's own labels, so the effect holds in part. The round-3 verifier reads the criterion as met on the clusters found here, the rule the attempt uses for every downstream claim, as the criterion has no partial clause; on the paper's own labels it would be not reproduced. The test on the CSD minimum centre (depth_csd_min) is not run here.

## fig6a-fs1-symmetric

> The spike propagation symmetry index separates FS1 (symmetric propagation) from FS2 and FS3
> (asymmetric propagation).

In [ ]:
for r in READINGS:
    k = three(T[r], "si", ["FS1", "FS2", "FS3"])
    ratio = T[r].PL_below / T[r].PL_above
    kr = W.kruskal_holm({g: ratio[T[r].six == g].values for g in ["FS1", "FS2", "FS3"]})
    print(f"{READINGS[r]:18s} SI medians {dict((g, round(v, 3)) for g, v in k['medians'].items())}, KW H = {k['H']:.1f}, "
          f"p = {k['p']:.2e}; Holm FS1-FS2 {k['post']['FS1-FS2']:.2e}, FS1-FS3 {k['post']['FS1-FS3']:.2e}; "
          f"ratio variant KW p = {kr['p']:.2e}")

**Result: reproduced, resting on an assumption.** FS1 has the lowest symmetry index (median 0.391 against 1.019 and 1.084, reading c, K forced to 3), Kruskal-Wallis p = 1.3e-09, Holm FS1 against FS2 p = 2.9e-06 and FS1 against FS3 p = 2.7e-08. Readings (a) and (b) give the same at p below 1e-09. On the Source Data labels FS1 against FS3 gives p = 0.0026, below 0.05 but not 0.001. The ratio variant separates the clusters less (p = 0.013, reading c).

## fig4g-experiment-classifier-fs-rs

> A linear SVM trained on TPW and REP of in vivo units discriminates FS from RS with a
> validation accuracy above 99%.

As Exp_Model_Classifier.ipynb cells 7 and 12: for each of 100 resamplings, FS units drawn with
replacement up to the RS count are added, TPW and REP are z-scored over the upsampled set,
and a linear SVC (C = 1, balanced weights) is fitted on 100 stratified 75/25 splits. The
resampling and the split seeds come from the claim's generator. The five seeds are run with
20 resamplings each, to keep the time down (a standard choice; the mean accuracy over 100
resamplings varies little).

In [ ]:
from sklearn import svm
from sklearn.model_selection import train_test_split
SEED_SVM = seed_of("fig4g-experiment-classifier-fs-rs")


def exp_classifier(df, lab, seed, n_rep=100, n_split=100, newX_raw=None, cols=("TPW", "REP"), model_name="svm"):
    """The code's procedure: upsample, z-score, then split (Exp_Model_Classifier.ipynb)."""
    rng = np.random.default_rng(seed)
    accs, votes = [], None
    for _ in range(n_rep):
        idx = W.upsample_index(lab, rng)
        Xu = df[list(cols)].values[idx]
        mu, sd_ = Xu.mean(0), Xu.std(0)
        acc, v, cls = W.classifier((Xu - mu) / sd_, lab[idx],
                                   None if newX_raw is None else (newX_raw - newX_raw.mean(0)) / newX_raw.std(0),
                                   model_name, n_split, int(rng.integers(2 ** 31)))
        accs.append(acc)
        if v is not None:
            votes = v if votes is None else votes + v
    return float(np.mean(accs)), votes, cls


def split_first(X, lab, seed, n_split=100, model_name="svm"):
    """Variant without leakage: split the units first, upsample and z-score the training part
    only, test on held-out units; returns the mean accuracy and the mean balanced accuracy."""
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import balanced_accuracy_score
    rng = np.random.default_rng(seed)
    acc, bacc = [], []
    for _ in range(n_split):
        tr, te = train_test_split(np.arange(len(lab)), test_size=0.25, stratify=lab, random_state=int(rng.integers(1, 2 ** 31 - 1)))
        ui = tr[W.upsample_index(lab[tr], rng)]
        mu, sd_ = X[ui].mean(0), X[ui].std(0)
        m = (svm.SVC(kernel="linear", C=1, class_weight="balanced") if model_name == "svm" else
             RandomForestClassifier(n_estimators=20, criterion="gini", max_features="sqrt", bootstrap=True, random_state=0, n_jobs=1))
        m.fit((X[ui] - mu) / sd_, lab[ui])
        pr = m.predict((X[te] - mu) / sd_)
        acc.append(np.mean(pr == lab[te]))
        bacc.append(balanced_accuracy_score(lab[te], pr))
    return float(np.mean(acc)), float(np.mean(bacc))


t_ = time.time()
acc_svm = {}
for r in READINGS:
    lab = labels[r].onech.values
    df = units.loc[labels[r].index]
    n_rep = 100 if r in ("c", "sd") else 20
    acc_svm[r], _, _ = exp_classifier(df, lab, SEED_SVM, n_rep=n_rep)
    print(f"{READINGS[r]:18s} mean validation accuracy FS against RS: {100 * acc_svm[r]:.2f}% ({n_rep} resamplings x 100 splits)")
print(f"({time.time() - t_:.0f} s)")
lab = labels["c"].onech.values
df = units.loc[labels["c"].index]
print("reading (c), five seeds (20 resamplings x 100 splits):",
      [round(100 * exp_classifier(df, lab, SEED_SVM + k * 1000, n_rep=20)[0], 2) for k in range(5)])
g_ = np.random.default_rng(SEED_SVM + 1)
perm = g_.permutation(lab)
print("chance baseline, reading (c), labels permuted (the code's procedure, 20 x 100): %.2f%%; majority class %.2f%%"
      % (100 * exp_classifier(df, perm, SEED_SVM + 2, n_rep=20)[0], 100 * max(np.mean(lab == "FS"), np.mean(lab == "RS"))))
for r in ("c", "sd"):
    lab_r = labels[r].onech.values
    X_r = units.loc[labels[r].index, ["TPW", "REP"]].values.astype(float)
    a1, b1 = split_first(X_r, lab_r, SEED_SVM + 3)
    a0, b0 = split_first(X_r, g_.permutation(lab_r), SEED_SVM + 4)
    print(f"{READINGS[r]:18s} split before upsampling (100 splits): accuracy {100 * a1:.2f}%, balanced {100 * b1:.2f}%; "
          f"labels permuted: accuracy {100 * a0:.2f}%, balanced {100 * b0:.2f}%")


**Result: reproduced.** The mean validation accuracy is 99.34% (reading c; 99.34 to 99.37% under the five seeds) and 98.69% on the Source Data labels. The code's procedure upsamples before it splits, so copies of a unit can be in training and test; for this linear SVM that does not inflate the score: with permuted labels it gives 53.18%, and splitting before upsampling gives 99.19% (balanced 99.44%) on reading (c), against a balanced 52.93% with permuted labels. The FS and RS labels come from K-means on the same two features, so a high accuracy is expected.

## fig5d-experiment-classifier-rs

> A classifier trained on the multi-channel in vivo features discriminates RS1 to RS3 with
> accuracy above 99%.

Random forest (20 trees, gini) on z-scored 1/V below and above of RS units, classes upsampled
to the largest, 100 stratified 75/25 splits for each of 100 resamplings (criterion). Units
whose 1/V above is NaN are left out (counted).

In [ ]:
SEED_RF = seed_of("fig5d-experiment-classifier-rs")


def rs_classifier(r, seed, n_rep=100, n_split=100, permute=False):
    rng = np.random.default_rng(seed)
    t = T[r][T[r].onech == "RS"].dropna(subset=["PL_below", "PL_above"])
    lab = t.six.values
    if permute:
        lab = rng.permutation(lab)
    accs = []
    for _ in range(n_rep):
        idx = W.upsample_index(lab, rng)
        X = W.zscore(t[["PL_below", "PL_above"]].values[idx])
        accs.append(W.classifier(X, lab[idx], None, "rf", n_split, int(rng.integers(2 ** 31)))[0])
    return float(np.mean(accs)), len(T[r][T[r].onech == "RS"]) - len(t)


t_ = time.time()
for r in READINGS:
    n_rep = 20 if r in ("c", "sd") else 5
    acc, dropped = rs_classifier(r, SEED_RF, n_rep=n_rep)
    print(f"{READINGS[r]:18s} RS1-3 random forest accuracy {100 * acc:.2f}% ({n_rep} resamplings x 100 splits), NaN units left out {dropped}")
print(f"({time.time() - t_:.0f} s)")
print("reading (c), five seeds (5 resamplings x 100 splits):",
      [round(100 * rs_classifier("c", SEED_RF + k * 1000, n_rep=5)[0], 2) for k in range(5)])
print("chance baseline, reading (c), labels permuted (the code's procedure, 5 x 100): %.2f%%"
      % (100 * rs_classifier("c", SEED_RF + 1, n_rep=5, permute=True)[0]))
for r in ("c", "sd"):
    t = T[r][T[r].onech == "RS"].dropna(subset=["PL_below", "PL_above"])
    a1, b1 = split_first(t[["PL_below", "PL_above"]].values.astype(float), t.six.values, SEED_RF + 2, model_name="rf")
    a0, b0 = split_first(t[["PL_below", "PL_above"]].values.astype(float), np.random.default_rng(SEED_RF + 3).permutation(t.six.values),
                         SEED_RF + 4, model_name="rf")
    print(f"{READINGS[r]:18s} split before upsampling (100 splits): accuracy {100 * a1:.2f}%, balanced {100 * b1:.2f}%; "
          f"labels permuted: accuracy {100 * a0:.2f}%, balanced {100 * b0:.2f}%")


**Result: partially reproduced.** The random forest reaches 98.55% on RS1 to RS3 of reading (c) (98.55 to 98.73% under the five seeds), above 95% but not 99%, and 93.27% on the Source Data labels. For the random forest the code's order (upsample, then split) inflates the score: permuted labels give 61.67% where chance is about 33%. Splitting before upsampling gives 97.53% (balanced 97.34%) on reading (c) and 84.55% on the Source Data labels, against 34.20% and 44.53% with permuted labels (balanced 31.58% and 37.09%).

## fig3a-isi-fs-rs

> During drifting gratings the ISI median of FS units is 19.63 ms (95% CI 19.60 to 19.67) and
> that of RS units 53.37 ms (95% CI 53.30 to 53.43), a significant difference.

Units with EAP amplitude above 50 uV and at least 100 spikes (Methods) during drifting
gratings. ISIs pooled over the units of a cluster, under (a) ISIs within each presentation and
(b) ISIs within each drifting-grating block. The 95% CI is a bootstrap of the median over ISIs
(200 resamples under the claim's seed).

In [ ]:
SEED_ISI = seed_of("fig3a-isi-fs-rs")


def active(df, nspk_col):
    return df[(df.A > 50) & (df[nspk_col] >= 100)]


def pooled(r, tag, grp):
    t = T[r]
    t = active(t, "n_dg" if tag == "a" else "n_dg_blocks")
    ids = t.index[t.onech == grp]
    key = "isi_a" if tag == "a" else "isi_b"
    sess = units.loc[ids, "session"]
    return np.concatenate([R[s][key][i] for i, s in zip(ids, sess)]).astype(np.float64) * 1000, len(ids)


rng = np.random.default_rng(SEED_ISI)
for r in READINGS:
    for tag in "ab":
        fs_isi, nfs = pooled(r, tag, "FS")
        rs_isi, nrs = pooled(r, tag, "RS")
        ci = {}
        for name, x in (("FS", fs_isi), ("RS", rs_isi)):
            bs = [np.median(x[rng.integers(0, len(x), len(x))]) for _ in range(200)]
            ci[name] = np.percentile(bs, [2.5, 97.5])
        p = stats.mannwhitneyu(fs_isi, rs_isi, alternative="two-sided").pvalue
        print(f"{READINGS[r]:18s} ({tag}) FS {nfs} units {len(fs_isi)} ISIs median {np.median(fs_isi):.2f} ms "
              f"[{ci['FS'][0]:.2f}, {ci['FS'][1]:.2f}]; RS {nrs} units {len(rs_isi)} ISIs median {np.median(rs_isi):.2f} ms "
              f"[{ci['RS'][0]:.2f}, {ci['RS'][1]:.2f}]; MW p = {p:.1e}")
fs_isi, _ = pooled("c", "a", "FS")
rs_isi, _ = pooled("c", "a", "RS")
meds = []
for k in range(5):
    g = np.random.default_rng(SEED_ISI + 1000 * k)
    meds.append([round(float(np.percentile([np.median(x[g.integers(0, len(x), len(x))]) for _ in range(50)], q)), 2)
                 for x in (fs_isi, rs_isi) for q in (2.5, 97.5)])
print("reading (c), (a): bootstrap CI bounds (FS low, FS high, RS low, RS high) under five seeds, 50 resamples:", meds)

**Result: partially reproduced, resting on an assumption.** Under reading (c), ISIs within presentations, the pooled median is 21.00 ms for FS and 55.33 ms for RS (Mann-Whitney p below the smallest number printed), against 19.63 and 53.37 ms: both within 10% (7.0% and 3.7%), neither in the paper's confidence interval. The block reading gives 21.70 and 64.83 ms. On the Source Data labels the medians are 20.50 and 54.67 ms; under reading (a) 22.67 and 52.93 ms (FS 15% off). The spike counts (1.33 and 1.91 million, reading c) are about half the paper's, as are the sessions. The bootstrap intervals are stable under the five seeds.

## fig3a-rs1-excitability

> RS1 units fire faster and with less variable spiking (lower CV and LV) than RS2 and RS3
> units during drifting gratings.

Units with EAP amplitude above 50 uV and at least 100 spikes during drifting gratings; rate,
CV and LV over the drifting-grating spikes under the two period readings; Kruskal-Wallis
with Holm-corrected Mann-Whitney tests. The Source Data per-unit values (sheet
Fig3a_rate_cv_lv) are compared with the values here on shared units.

In [ ]:
def rs_tests(r, tag, cols):
    t = active(T[r], "n_dg" if tag == "a" else "n_dg_blocks")
    out = {}
    for c in cols:
        k = three(t, c, ["RS1", "RS2", "RS3"])
        out[c] = k
    return out, t
for r in READINGS:
    for tag in "ab":
        cols = {"rate": "rate_dg" if tag == "a" else "rate_dg_blocks", "cv": f"cv_dg_{tag}", "lv": f"lv_dg_{tag}"}
        res, t = rs_tests(r, tag, cols.values())
        n = t.six.value_counts().reindex(["RS1", "RS2", "RS3"]).to_dict()
        for name, c in cols.items():
            k = res[c]
            print(f"{READINGS[r]:18s} ({tag}) {name:4s} medians {dict((g, round(v, 3)) for g, v in k['medians'].items())}, "
                  f"KW p = {k['p']:.1e}, Holm RS1-RS2 {k['post']['RS1-RS2']:.1e}, RS1-RS3 {k['post']['RS1-RS3']:.1e}")
        print(f"   units {n}")
m3 = sd_table("Fig3a_rate_cv_lv").set_index("unit_id")
sh = m3.index.intersection(units.index)
for a, b in [("rate_dg", "Rate"), ("rate_dg_blocks", "Rate"), ("cv_dg_a", "CV"), ("cv_dg_b", "CV"), ("lv_dg_a", "LV"), ("lv_dg_b", "LV")]:
    d = np.abs(units.loc[sh, a].astype(float) - m3.loc[sh, b].astype(float))
    print(f"{a:15s} against Source Data {b}: {len(sh)} units, median |diff| {d.median():.3f}, "
          f"Spearman r {stats.spearmanr(units.loc[sh, a], m3.loc[sh, b].astype(float)).statistic:.3f}")
print("Source Data units of Fig3a_rate_cv_lv in the held sessions:", len(sh), "of", len(m3),
      "; by cluster here (Source Data labels):", labels["sd"].loc[sh].six.value_counts().to_dict())

**Result: partially reproduced, resting on an assumption.** Under reading (c) one of the three measures holds under each period reading. Over the drifting-grating blocks (the period whose CV matches the Source Data best, Spearman 0.848 against 0.502), RS1 has the lower CV (1.54 against 1.76 and 1.68; omnibus p = 1.8e-05, Holm 4.5e-05 and 0.0035), but the rate (Holm p = 0.051 and 0.051) and LV (0.048 and 0.054) miss in one comparison each. Within presentations only the rate holds (3.92 against 2.56 and 2.26 Hz, Holm p = 0.032 and 0.037). Reading (a) gives all three under the block reading (each Holm p below 0.02); on the Source Data labels RS1 differs from RS3 only in CV. The unit counts (166, 75, 87) come from 13 of 25 sessions.

## fig3a-rs23-visual-responses

> RS2 and RS3 have higher f1/f0, modulation index and lifetime sparseness to drifting
> gratings than RS1.

The three metrics as AllenSDK defines them (helper `dg_metrics`, which reproduces the Source
Data values on shared units, below). Units as fig3a-rs1-excitability.

In [ ]:
ms = sd_table("Fig3a_stimulus_metrics").set_index("unit_id")
sh = ms.index.intersection(units.index)
for c in ["f1_f0_dg", "mod_idx_dg", "lifetime_sparseness_dg"]:
    d = np.abs(units.loc[sh, c].astype(float) - ms.loc[sh, c].astype(float))
    print(f"{c}: {len(sh)} shared units, equal (<1e-6) for {np.mean(d < 1e-6):.1%}, max |diff| {d.max():.2e}")
VIS = ["f1_f0_dg", "mod_idx_dg", "lifetime_sparseness_dg"]
for r in READINGS:
    t = active(T[r], "n_dg")
    for c in VIS:
        k = three(t, c, ["RS1", "RS2", "RS3"])
        print(f"{READINGS[r]:18s} {c:23s} medians {dict((g, round(v, 3)) for g, v in k['medians'].items())}, KW p = {k['p']:.1e}, "
              f"Holm RS1-RS2 {k['post']['RS1-RS2']:.1e}, RS1-RS3 {k['post']['RS1-RS3']:.1e}")
    print("   units", t.six.value_counts().reindex(["RS1", "RS2", "RS3"]).to_dict())

**Result: not reproduced, resting on an assumption.** With the AllenSDK metrics (which reproduce the Source Data values exactly for 98.9 to 100% of shared units), RS1 has the lowest f1/f0 and lifetime sparseness under reading (c), but in each the RS1 against RS3 test fails (Holm p = 0.056 and 0.065), and the modulation index does not differ (omnibus p = 0.21). On the Source Data labels also only the RS1 against RS2 comparisons hold. Under reading (a), f1/f0 and lifetime sparseness hold in full (Holm p below 0.015), which would make the claim partially reproduced.

## fig3a-fs-visual-absent

> FS1 to FS3 do not differ significantly in their visual responses to drifting gratings.

In [ ]:
for r in READINGS:
    t = active(T[r], "n_dg")
    for c in VIS:
        k = three(t, c, ["FS1", "FS2", "FS3"])
        print(f"{READINGS[r]:18s} {c:23s} medians {dict((g, round(v, 3)) for g, v in k['medians'].items())}, KW H = {k['H']:.2f}, p = {k['p']:.3f}")
    print("   units", t.six.value_counts().reindex(["FS1", "FS2", "FS3"]).to_dict())

**Result: reproduced.** No metric differs across FS1 to FS3 (Kruskal-Wallis p = 0.833, 0.887 and 0.969, reading c; all p above 0.13 under readings a and b and the Source Data labels).

## fig3c-fs-more-phase-locked

> During drifting gratings a larger share of FS than RS units is phase-locked to the LFP in
> every band (theta 96.6% against 86.7%, alpha 86.7% against 63.9%, beta 88.2% against 67.7%,
> low gamma 83.3% against 58.1%, high gamma 91.1% against 68.5%).

Units with EAP amplitude above 50 uV and at least 100 spikes during drifting gratings, at
most 10,000 spikes; Hilbert phase of the band-passed LFP (5th-order Butterworth, zero phase);
phase-locked when the Rayleigh p < 0.001 (Methods). Spikes are those inside the presentations;
the LFP is filtered over each whole block (a standard choice, to keep filter edges away from
the spikes). The readings of the criterion: LFP channel (a) the nearest other than the peak
channel, (b) the nearest on the other side; a third channel, (c) the nearest at least 100 um
from the peak channel, checks that the unit's own spikes do not carry the locking (not a
reading of the criterion); the cap (first) the first 10,000 spikes, (random)
a random 10,000; band edges of the Results or of the Methods (alpha and beta). Two-sample z
tests for proportions, Holm-corrected over the five bands.

In [ ]:
PH = pd.concat([R[s]["phase"].assign(session=s) for s in ALL_SESSIONS], ignore_index=True)
BANDS = ["theta", "alpha", "beta", "low_gamma", "high_gamma"]
EXP = {"theta": (96.6, 86.7), "alpha": (86.7, 63.9), "beta": (88.2, 67.7), "low_gamma": (83.3, 58.1),
       "high_gamma": (91.1, 68.5)}


def phase_table(period, lfp, cap, bandset):
    """One row per unit and band, for the chosen readings; the Methods band set replaces only
    alpha and beta."""
    p = PH[(PH.period == period) & (PH.lfp == lfp) & (PH.cap == cap)]
    res = p[p.bands == "results"]
    if bandset == "methods":
        res = pd.concat([res[~res.band.isin(["alpha", "beta"])], p[p.bands == "methods"]])
    return res


def eligible(t, r, nspk="n_dg"):
    tt = T[r]
    ok = tt.index[(tt.A > 50) & (tt[nspk] >= 100)]
    return t[t.unit_id.isin(ok)].merge(tt[["onech", "six", "depth_csd", "session"]], left_on="unit_id",
                                       right_index=True, suffixes=("", "_u"))


def percent_locked(t):
    out = {}
    for b in BANDS:
        x = t[t.band == b]
        k = {g: (int((x[x.onech == g].p < 0.001).sum()), int((x.onech == g).sum())) for g in ("FS", "RS")}
        out[b] = k
    return out


combos = [(lfp, cap, bs) for lfp in "abc" for cap in ("first", "random0") for bs in ("results", "methods")]
pl_summary = []
for r in READINGS:
    for lfp, cap, bs in combos:
        t = eligible(phase_table("dg", lfp, cap, bs), r)
        pc = percent_locked(t)
        zp = [W.two_prop_z(*pc[b]["FS"], *pc[b]["RS"]) for b in BANDS]
        adj = W.holm([z[1] for z in zp])
        row = {"labels": r, "lfp": lfp, "cap": cap, "bands": bs,
               "n_FS": pc["theta"]["FS"][1], "n_RS": pc["theta"]["RS"][1]}
        for b, a_ in zip(BANDS, adj):
            fs_, rs_ = (100 * pc[b][g][0] / pc[b][g][1] for g in ("FS", "RS"))
            row[f"{b}_FS"], row[f"{b}_RS"], row[f"{b}_pHolm"] = round(fs_, 1), round(rs_, 1), float("%.1e" % a_)
        row["bands FS>RS p<0.05"] = sum(row[f"{b}_FS"] > row[f"{b}_RS"] and row[f"{b}_pHolm"] < 0.05 for b in BANDS)
        row["FS>RS all, p<0.05"] = all(row[f"{b}_FS"] > row[f"{b}_RS"] and row[f"{b}_pHolm"] < 0.05 for b in BANDS)
        row["all within tol"] = all(abs(row[f"{b}_FS"] - EXP[b][0]) <= 0.1 * EXP[b][0] and
                                    abs(row[f"{b}_RS"] - EXP[b][1]) <= 0.1 * EXP[b][1] for b in BANDS)
        pl_summary.append(row)
pl_summary = pd.DataFrame(pl_summary)
print(pl_summary.to_string(index=False))
print("LFP (c), the nearest LFP channel at least 100 um from the peak channel, against LFP (a), reading (c), first spikes, Results bands:")
print(pl_summary[(pl_summary.labels == "c") & (pl_summary.cap == "first") & (pl_summary.bands == "results")][["lfp"] + [f"{b}_{g}" for b in BANDS for g in ("FS", "RS")]].to_string(index=False))
# seed: the random cap under the five seeds, reading (c), LFP (a), Results bands
for k in range(5):
    t = eligible(phase_table("dg", "a", f"random{k}", "results"), "c")
    pc = percent_locked(t)
    print(f"seed +{1000 * k}:", {b: (round(100 * pc[b]['FS'][0] / pc[b]['FS'][1], 1), round(100 * pc[b]['RS'][0] / pc[b]['RS'][1], 1)) for b in BANDS})
print("units per session (reading c, FS/RS):", eligible(phase_table("dg", "a", "first", "results"), "c")
      .query("band == 'theta'").groupby(["session", "onech"]).size().unstack().to_dict())

**Result: not reproduced, resting on an assumption.** Under reading (c), in every combination of the LFP, cap and band readings, a larger share of FS than RS units is phase-locked in all five bands (LFP a, first spikes: theta 98.0 against 96.3%, alpha 85.7 against 82.3%, beta 94.9 against 84.7%, low gamma 91.8 against 77.4%, high gamma 93.9 against 85.6%), but the Holm-corrected z test gives p < 0.05 in only two bands (beta and low gamma), with 98 FS and 327 RS units. Several RS percentages are above the paper's tolerance (alpha 82.3 against 63.9 +- 6.39). Reading (a) (115 FS, 485 RS) gives 3 or 4 significant bands under LFP readings (a) and (b), which would make it partially reproduced (5 only with LFP c, which is not a reading of the criterion); the Source Data labels give 2 or 3.

## fig3c-rs-prefers-theta

> RS phase-locking is most common for theta and less common at higher frequencies (beta, low
> and high gamma), while the share of phase-locked FS units stays high across bands.

In [ ]:
for _, row in pl_summary.iterrows():
    rs_ = [row[f"{b}_RS"] for b in BANDS]
    fs_ = [row[f"{b}_FS"] for b in BANDS]
    ok = (rs_[0] == max(rs_) and all(rs_[0] > rs_[i] for i in (2, 3, 4)) and np.ptp(fs_) < np.ptp(rs_))
    print(f"{row['labels']:2s} lfp {row['lfp']} {row['cap']:7s} {row['bands']:7s} RS {rs_} FS {fs_}: "
          f"FS range {np.ptp(fs_):.1f}, RS range {np.ptp(rs_):.1f}, holds {ok}")

**Result: reproduced.** In every combination of readings the RS theta percentage is the highest (96.3% against 77.4 to 85.6% for beta and the gammas, reading c, LFP a), and the FS percentages vary over a smaller range than the RS ones (12.3 against 18.9 points). LFP (c), 100 um or more from the peak channel, gives the same (10.2 against 31.1 points).

## figs7-spontaneous-less-locked

> Far fewer FS and RS units are phase-locked during spontaneous activity than during drifting
> gratings, in every band.

The same method on spikes and LFP of the gray-screen intervals of 30 s or more, for units with
at least 100 spikes in both periods (and EAP amplitude above 50 uV).

In [ ]:
for r in READINGS:
    for lfp, cap, bs in combos[:4] + combos[8:10]:
        tt = T[r]
        ok = tt.index[(tt.A > 50) & (tt.n_dg >= 100) & (tt.n_spont >= 100)]
        out = {}
        for period in ("dg", "spont"):
            t = eligible(phase_table(period, lfp, cap, bs), r)
            out[period] = percent_locked(t[t.unit_id.isin(ok)])
        line = {b: tuple(round(100 * out[p][b][g][0] / max(out[p][b][g][1], 1), 1) for p in ("spont", "dg") for g in ("FS", "RS")) for b in BANDS}
        lower = all(out["spont"][b][g][0] / out["spont"][b][g][1] < out["dg"][b][g][0] / out["dg"][b][g][1]
                    for b in BANDS for g in ("FS", "RS"))
        print(f"{r:2s} lfp {lfp} {cap:7s} {bs:7s} n FS {out['spont']['theta']['FS'][1]} RS {out['spont']['theta']['RS'][1]}; "
              f"(spont FS, spont RS, dg FS, dg RS) % {line}; spontaneous lower everywhere: {lower}")

**Result: partially reproduced.** For units active in both periods, fewer units are phase-locked during the gray screen than during gratings in every band, for FS and RS (reading c), but far more than the paper reports: theta 90.8% of FS and 77.7% of RS against 19.5 and 4.7%. The LFP channel 100 um or more from the peak channel gives as many (theta 92.9 and 81.0%), so the unit's own spike waveform on the nearest channel does not explain the high spontaneous locking; it does raise the RS high-gamma share (80.1% with LFP a, 46.8% with LFP c).

## fig3c-fs-stronger-earlier

> FS units phase-lock more strongly (higher kappa) than RS units across bands and spike
> earlier in the cycle than RS units in beta and low gamma.

Kappa and preferred phase of each phase-locked unit and band. Readings of kappa: (kappa) the
pycircstat von Mises estimate, (R) the mean resultant length. "Earlier in the cycle" is read
as a lower preferred phase (degrees from 0 to 360, 180 at the trough). Two-sided Mann-Whitney
tests, Holm-corrected over the five bands. Shown for LFP (a) and (b), first 10,000 spikes and
Results band edges; the other cap and band readings change the percentages above by at most
a few units.

In [ ]:
def locked(period, lfp, r, cap="first", bs="results"):
    t = eligible(phase_table(period, lfp, cap, bs), r)
    return t[t.p < 0.001]


def compare(t, col, a="FS", b="RS", grp="onech", bands=BANDS):
    raw, med = [], {}
    for band in bands:
        x = t[t.band == band]
        va, vb = x.loc[x[grp] == a, col].values, x.loc[x[grp] == b, col].values
        raw.append(stats.mannwhitneyu(va, vb, alternative="two-sided").pvalue if len(va) and len(vb) else np.nan)
        med[band] = (round(float(np.median(va)), 3) if len(va) else np.nan, round(float(np.median(vb)), 3) if len(vb) else np.nan)
    return med, W.holm(raw), raw


for r in READINGS:
    for lfp in "ab":
        t = locked("dg", lfp, r)
        for col in ("kappa", "R", "phase"):
            med, adj, _ = compare(t, col)
            print(f"{READINGS[r]:18s} LFP {lfp} {col:5s} (FS, RS) medians {med}; Holm p {[float('%.1e' % x) for x in adj]}")

**Result: not reproduced.** Among phase-locked units FS kappa is higher than RS in all five bands (reading c, LFP a), but only low and high gamma differ after Holm correction (p = 0.034 and 2.4e-06; theta, alpha and beta p = 0.072, 0.12, 0.072). The FS preferred phase is not earlier in beta (176.1 against 174.3 degrees) and is earlier in low gamma without significance (178.1 against 184.8, Holm p = 1.0). The resultant length gives the same tests. Under LFP (b) only high gamma kappa differs after Holm correction (p = 0.0011; low gamma p = 0.091).

## fig3d-rs2-gamma-locking

> Among RS clusters, RS2 phase-locks more strongly to low and high gamma than RS1, while the
> preferred phase of RS1 to RS3 stays at about 180 to 200 degrees.

In [ ]:
def circ_mean_deg(x):
    return float(np.degrees(np.angle(np.mean(np.exp(1j * np.radians(x))))) % 360)


for r in READINGS:
    for lfp in "ab":
        t = locked("dg", lfp, r)
        for col in ("kappa", "R"):
            out = []
            for band in ("low_gamma", "high_gamma"):
                x = t[t.band == band]
                k = three(x, col, ["RS1", "RS2", "RS3"])
                out.append(f"{band}: medians {dict((g, round(v, 3)) for g, v in k['medians'].items())}, KW p {k['p']:.1e}, Holm RS1-RS2 {k['post']['RS1-RS2']:.1e}")
            print(f"{READINGS[r]:18s} LFP {lfp} {col}: " + "; ".join(out))
        ph = {band: {g: round(circ_mean_deg(t[(t.band == band) & (t.six == g)].phase), 1) for g in ("RS1", "RS2", "RS3")} for band in BANDS}
        allv = [v for d in ph.values() for v in d.values()]
        print(f"   mean phases {ph}; range {min(allv):.1f} to {max(allv):.1f}; within 162 to 220: {all(162 <= v <= 220 for v in allv)}")
        print("   units per cluster (theta):", t[t.band == "theta"].six.value_counts().reindex(["RS1", "RS2", "RS3"]).to_dict())

**Result: not reproduced.** RS1 to RS3 do not differ in low or high gamma kappa (Kruskal-Wallis p = 0.35 and 0.78, reading c, LFP a), and RS2 is not above RS1. The cluster mean phases lie between 162 and 220 degrees under every reading.

## fig3e-fs-cluster-locking

> FS3 phase-locks more strongly to high gamma than FS2, and FS1 to FS3 have distinct preferred
> phases in alpha, beta and low gamma.

In [ ]:
for r in READINGS:
    for lfp in "ab":
        t = locked("dg", lfp, r)
        x = t[t.band == "high_gamma"]
        out = []
        for col in ("kappa", "R"):
            k = three(x, col, ["FS1", "FS2", "FS3"])
            out.append(f"high gamma {col} medians {dict((g, round(v, 3)) for g, v in k['medians'].items())}, Holm FS2-FS3 {k['post']['FS2-FS3']:.1e}")
        ph = []
        for band in ("alpha", "beta", "low_gamma"):
            k = three(t[t.band == band], "phase", ["FS1", "FS2", "FS3"])
            ph.append(f"{band} KW p {k['p']:.1e} medians {dict((g, round(v, 1)) for g, v in k['medians'].items())}")
        print(f"{READINGS[r]:18s} LFP {lfp}: " + "; ".join(out))
        print("   phase: " + "; ".join(ph))
        print("   units (theta):", t[t.band == "theta"].six.value_counts().reindex(["FS1", "FS2", "FS3"]).to_dict(),
              "; expected 100, 54, 49 of the 25 sessions")

**Result: partially reproduced, resting on an assumption.** On FS clusters with K forced to 3 (reading c), FS3 does not lock more strongly than FS2 in high gamma (Holm p = 1.0), and the preferred phases differ in alpha (Kruskal-Wallis p = 0.046) but not in beta or low gamma (p = 0.24 and 0.96). The Source Data labels give the same kappa result.

## figs6c-fs-stronger-across-depth

> FS units phase-lock more strongly than RS units in the supragranular, granular and
> infragranular regions in all LFP bands.

Regions: reading (a) of the criterion, the CCF layer of the peak channel, cannot be run, as the
electrodes tables give the structure (VISp) but no layer. Reading (b), depth relative to the
CSD layer 4 centre, granular within 100 um of it, supragranular above, infragranular below.
The centre is the CSD centre computed here (depth_csd, see figs5-depth-distribution), the
method the criterion names; the layer 4 centre of the code's unit tables is shown as a variant.
Coupling read as (i) kappa, Mann-Whitney, (ii) percentage phase-locked, z test Holm-corrected
over the 15 region and band cells. LFP (a), first 10,000 spikes, Results bands.

In [ ]:
def region(depth):
    return np.where(depth > 100, "supra", np.where(depth < -100, "infra", "granular"))


REG = ["supra", "granular", "infra"]


def depth_cells(r, depth_source="csd", lfp="a"):
    t = eligible(phase_table("dg", lfp, "first", "results"), r)
    d = t.depth_csd if depth_source == "csd" else t.unit_id.map(units.depth_code)
    t = t.assign(region=region(d.astype(float)))[d.notna().values]
    return t


for r in READINGS:
    for src in ("code", "csd"):
        t = depth_cells(r, src)
        n = t[t.band == "theta"].groupby(["region", "onech"]).size().unstack().reindex(REG)
        lk = t[t.p < 0.001]
        cells_k, cells_p, raw_z = [], [], []
        for reg in REG:
            for band in BANDS:
                x = lk[(lk.region == reg) & (lk.band == band)]
                fs_, rs_ = x[x.onech == "FS"].kappa, x[x.onech == "RS"].kappa
                pk = stats.mannwhitneyu(fs_, rs_, alternative="two-sided").pvalue if len(fs_) and len(rs_) else np.nan
                cells_k.append((reg, band, fs_.median() > rs_.median(), pk))
                y = t[(t.region == reg) & (t.band == band)]
                k1, n1 = int((y[y.onech == "FS"].p < 0.001).sum()), int((y.onech == "FS").sum())
                k2, n2 = int((y[y.onech == "RS"].p < 0.001).sum()), int((y.onech == "RS").sum())
                raw_z.append(W.two_prop_z(k1, n1, k2, n2)[1] if n1 and n2 else np.nan)
                cells_p.append((reg, band, (k1 / n1 if n1 else np.nan) > (k2 / n2 if n2 else np.nan)))
        adj = W.holm(np.nan_to_num(raw_z, nan=1.0))
        nk = sum(c[2] for c in cells_k)
        nks = sum(c[2] and c[3] < 0.05 for c in cells_k)
        npc = sum(c[2] for c in cells_p)
        nps = sum(c[2] and a_ < 0.05 for c, a_ in zip(cells_p, adj))
        print(f"{READINGS[r]:18s} depth {src}: units {n.to_dict()}")
        print(f"   (i) kappa FS > RS in {nk} of 15 cells, with p < 0.05 in {nks}; (ii) % locked FS > RS in {npc} of 15, Holm p < 0.05 in {nps}")
        print("   kappa cells:", [(c[0], c[1], bool(c[2]), float("%.1e" % c[3])) for c in cells_k])
print("legend counts: supragranular 37 FS and 107 RS, granular 54 FS and 185 RS, infragranular 112 FS and 453 RS")

**Result: not reproduced, resting on an assumption.** With regions from the CSD layer 4 centre computed here (reading b of the criterion; the CCF layer cannot be read, see the data check) and the units of reading (c), the percentage of phase-locked FS units exceeds that of RS units in 14 of 15 region and band cells (none with Holm p below 0.05), and kappa is higher for FS in 12 of 15 (5 with p below 0.05). With the code's layer 4 centre the percentage is higher in 13 of 15 and kappa in 12 of 15 (8 with p below 0.05). Reading (a) with the CSD centre gives the percentage higher in all 15 cells (3 with Holm p below 0.05), which would make it partially reproduced. Region counts (reading c, CSD centre) are 19, 32, 47 FS and 48, 106, 173 RS (code's centre 16, 34, 42 and 60, 101, 155), against the legend's 37, 54, 112 and 107, 185, 453 for 25 sessions.

## figs6d-rs3-supragranular-beta

> RS1 to RS3 differ most in coupling strength (kappa) in supragranular beta and low gamma, and
> in supragranular beta the kappa of RS3 is almost double that of RS1.

In [ ]:
for r in READINGS:
    for src in ("code", "csd"):
        lk = depth_cells(r, src)
        lk = lk[(lk.p < 0.001) & (lk.region == "supra")]
        out = []
        for band in ("beta", "low_gamma"):
            x = lk[lk.band == band]
            a3, a1, a2 = (x[x.six == g].kappa for g in ("RS3", "RS1", "RS2"))
            p = stats.mannwhitneyu(a3, a1, alternative="two-sided").pvalue if len(a3) and len(a1) else np.nan
            out.append(f"{band}: mean kappa RS1 {a1.mean():.3f}, RS2 {a2.mean():.3f}, RS3 {a3.mean():.3f}, "
                       f"ratio RS3/RS1 {a3.mean() / a1.mean():.2f}, MW RS3-RS1 p {p:.2e}, n {len(a1)}/{len(a2)}/{len(a3)}")
        print(f"{READINGS[r]:18s} depth {src}: " + "; ".join(out))

**Result: not reproduced, resting on an assumption.** In the supragranular region, with the CSD centre computed here and reading (c), the ratio of RS3 to RS1 mean kappa in beta is 1.14 (Mann-Whitney p = 0.48) and in low gamma 1.02 (p = 0.91), against 2 +- 0.2. With the code's layer 4 centre the ratios are 1.31 (p = 0.090) and 1.22 (p = 0.18). Reading (a) with the code's centre gives a beta ratio of 1.26 with p = 0.045 and low gamma p = 0.014, which meets the criterion's partial clause (RS3 above RS1 significantly in beta, ratio outside the tolerance); with the CSD centre reading (a) gives a beta ratio of 1.15 (p = 0.22).

## figs6c-granular-phase

> FS and RS spike at broadly the same LFP phase across depth, except in the granular region,
> where FS spikes earlier than RS in beta and gamma.

In [ ]:
for r in READINGS:
    for src in ("code", "csd"):
        lk = depth_cells(r, src)
        lk = lk[lk.p < 0.001]
        for reg in REG:
            med, _, raw = compare(lk[lk.region == reg], "phase", bands=["beta", "low_gamma", "high_gamma"])
            print(f"{READINGS[r]:18s} depth {src} {reg:8s} phase (FS, RS) medians {med}; MW p {[float('%.1e' % x) for x in raw]}")

**Result: not reproduced, resting on an assumption.** In the granular region, with the CSD centre computed here and reading (c), FS and RS phases do not differ in beta, low or high gamma (p = 0.72, 0.35 and 0.53), while the infragranular region shows a high-gamma difference (p = 0.027). With the code's layer 4 centre the granular tests give p = 0.95, 0.31 and 0.098 (infragranular high gamma p = 0.0012). No isolation reading or label set gives a granular phase difference with p < 0.05 under either centre.

## fig7-opto-unit-counts

> Opto-tagging found 25 well-isolated Pvalb units in 8 Pvalb-Cre mice and 18 Sst units in 12
> Sst-Cre mice.

10 ms square pulses; spikes 2 to 8 ms after onset; tagged when the trial-averaged rate there
is above 25 Hz and 2.5 times the rate before onset, with the baseline (a) -6 to 0 ms or
(b) -10 to 0 ms; isolation readings (a) and (b) of fig1d-unit-count, and (c) as above. The dandiset holds 5 of
the 8 Pvalb mice and 6 of the 12 Sst mice, so the counts per mouse are compared (25/8 = 3.1
or 24/8 = 3.0 Pvalb units, 18/12 = 1.5 Sst units per mouse).

In [ ]:
def tagged(df, base):
    return (df.opto_resp > 25) & (df.opto_resp > 2.5 * df[base])


def unit_ok(df):
    """The code's per-unit exclusions of the wild-type units (cell 8): peak channel at the
    centre, trough time within 0.1 ms, amplitude 15 to 300 uV (the 3 SD outlier step is a
    property of the pooled wild-type units and is not applied)."""
    return ~df.peak_off_centre.astype(bool) & (np.abs(df.neg_time) <= 0.1) & (df.A >= 15) & (df.A <= 300)


rows = []
for iso in "cab":
    for base in ("opto_base6", "opto_base10"):
        for g in ("Pvalb", "Sst"):
            d = units[(units.group == g) & units[f"iso_{iso}"]]
            tg = tagged(d, base)
            per = tg.groupby(d.session).sum().reindex(S[g]).astype(int)
            rows.append({"isolation": iso, "baseline": base[-2:].strip("e") + " ms", "line": g, "units": int(tg.sum()),
                         "mice": len(S[g]), "per mouse": round(tg.sum() / len(S[g]), 2), "per session": per.tolist(),
                         "after exclusions": int((tg & unit_ok(d)).sum())})
print(pd.DataFrame(rows).to_string(index=False))
d = units[units.group == "wt"]
for iso in "ca":
    print(f"wild-type sessions, units passing the same rule (false-positive check), reading ({iso}), 6 ms:",
          int(tagged(d[d[f"iso_{iso}"]], "opto_base6").sum()), "of", int(d[f"iso_{iso}"].sum()))
OPTO_ISO, OPTO_BASE, OPTO_EXCL = "c", "opto_base6", True
OPTO = units[units.group.isin(["Pvalb", "Sst"]) & units[f"iso_{OPTO_ISO}"] & tagged(units, OPTO_BASE)].copy()
OPTO["si"] = W.symmetry_index(OPTO.PL_below, OPTO.PL_above)
print("tagged units, isolation c, baseline 6 ms:", OPTO.group.value_counts().to_dict(),
      "; failing the per-unit exclusions:", OPTO[~unit_ok(OPTO)].groupby("group").size().to_dict(),
      "(peak channel off centre:", int(OPTO.peak_off_centre.sum()), "); NaN 1/V above:", int(OPTO.PL_above.isna().sum()))
if OPTO_EXCL:
    OPTO = OPTO[unit_ok(OPTO)].copy()
print("tagged units used below (isolation c, baseline 6 ms, per-unit exclusions):", OPTO.group.value_counts().to_dict())


**Result: not reproduced.** In the 5 Pvalb and 6 Sst sessions held, isolation (a) with the 6 ms baseline tags 13 Pvalb and 8 Sst units, 2.60 and 1.33 per mouse, against 3.0 to 3.1 and 1.5; isolation (b) gives 4.80 and 2.33 per mouse (6 ms) or 5.00 and 2.00 (10 ms), and reading (c) 2.40 and 1.17. Two Pvalb sessions have no tagged unit. The same rule tags 6 of 900 wild-type units (reading a). Of the 12 Pvalb and 7 Sst units of reading (c), 2 and 2 fail the code's per-unit exclusions (4 have the peak channel off centre).

## fig7d-pvalb-fs-sst-diffuse

> In one-channel feature space opto-tagged Pvalb units overlap the wild-type FS cluster, while
> Sst units are more diffuse and span FS and RS.

Readings (criterion): (a) the experiment-based classifier of fig4g (trained on the wild-type
units of reading c; tagged units z-scored with the wild-type upsampled set's mean and SD),
(b) the nearer of the two wild-type K-means centroids in the same z-scored space.

In [ ]:
wt = units.loc[labels["c"].index]
lab_wt = labels["c"].onech.values
rng = np.random.default_rng(seed_of("fig7d-pvalb-fs-sst-diffuse"))
votes = np.zeros((len(OPTO), 2), int)
for _ in range(20):
    idx = W.upsample_index(lab_wt, rng)
    Xu = wt[["TPW", "REP"]].values[idx]
    mu, sdv = Xu.mean(0), Xu.std(0)
    acc, v, cls = W.classifier((Xu - mu) / sdv, lab_wt[idx], (OPTO[["TPW", "REP"]].values - mu) / sdv, "svm", 20,
                               int(rng.integers(2 ** 31)))
    votes += v
OPTO["fsrs_svm"] = np.array(cls)[votes.argmax(1)]
Xw = W.pre_process(wt[["TPW", "REP"]].values)
mu, sdv = wt[["TPW", "REP"]].values.mean(0), wt[["TPW", "REP"]].values.std(0)
cent = {g: Xw[lab_wt == g].mean(0) for g in ("FS", "RS")}
Xo = (OPTO[["TPW", "REP"]].values - mu) / sdv
OPTO["fsrs_centroid"] = np.where(np.linalg.norm(Xo - cent["FS"], axis=1) < np.linalg.norm(Xo - cent["RS"], axis=1), "FS", "RS")
for c in ("fsrs_svm", "fsrs_centroid"):
    tab = pd.crosstab(OPTO.group, OPTO[c])
    print(c, tab.to_dict(), "; share in RS:", (tab["RS"] / tab.sum(1)).round(3).to_dict() if "RS" in tab else 0)
print("(20 resamplings x 20 splits)")
print(OPTO.groupby("group")[["TPW", "REP"]].median().round(3).to_dict())

**Result: reproduced.** All 10 Pvalb units fall in FS by both readings, while the Sst units fall in both clusters (2 FS, 3 RS); 60% of Sst units are in RS against 0% of Pvalb units.

## fig7e-pvalb-sst-one-channel

> Opto-tagged Pvalb and Sst units differ in TPW, REP and spike rate, Pvalb units having
> narrower waveforms, shorter REP and higher rates.

Two-sided Mann-Whitney tests; rate under the four period readings.

The tagged set used for the opto-tagged claims is isolation (c), the 6 ms baseline, and the
code's per-unit exclusions of the wild-type units (peak channel at the centre, trough time
within 0.1 ms, amplitude 15 to 300 uV), so that the opto-tagged and wild-type units pass the
same steps (an assumption); every other tagged set is shown in the readings table below.

In [ ]:
pv, ss = OPTO[OPTO.group == "Pvalb"], OPTO[OPTO.group == "Sst"]
for c in ["TPW", "REP", "rate_spont5", "rate_spont", "rate_first5min", "rate_session", "rate_dg"]:
    p = stats.mannwhitneyu(pv[c], ss[c], alternative="two-sided").pvalue
    print(f"{c:15s} median Pvalb {pv[c].median():.3f}, Sst {ss[c].median():.3f}, MW p = {p:.2e}  (n {len(pv)}, {len(ss)})")
print("per session (Pvalb, Sst units):", OPTO.groupby(["group", "session"]).size().to_dict())

## fig7g-pvalb-more-symmetric

> Opto-tagged Pvalb units have more symmetric EAP propagation (a lower symmetry index) than Sst
> units, as the models predict.

In [ ]:
a_, b_ = pv.si.dropna(), ss.si.dropna()
print(f"symmetry index mean Pvalb {a_.mean():.3f} (n {len(a_)}), Sst {b_.mean():.3f} (n {len(b_)}); "
      f"t-test p = {stats.ttest_ind(a_, b_).pvalue:.3f}; Mann-Whitney p = {stats.mannwhitneyu(a_, b_, alternative='two-sided').pvalue:.3f}")
sd7 = sd_table("Fig7deg")
print("Source Data Fig7deg: mean SI Pvalb", round(sd7[sd7.cell_type == "Pvalb"]["symmetry index"].astype(float).mean(), 3),
      "Sst", round(sd7[sd7.cell_type == "Sst"]["symmetry index"].astype(float).mean(), 3))

## fig7h-pvalb-higher-cv

> During drifting gratings opto-tagged Pvalb units have higher spike time variability (CV)
> than Sst units.

In [ ]:
for c in ("cv_dg_a", "cv_dg_b"):
    p = stats.mannwhitneyu(pv[c].dropna(), ss[c].dropna(), alternative="two-sided").pvalue
    print(f"{c}: median Pvalb {pv[c].median():.3f}, Sst {ss[c].median():.3f}, Pvalb higher: {pv[c].median() > ss[c].median()}, MW p = {p:.3f}")
sd7h = sd_table("Fig7h")
print("Source Data Fig7h CV medians: Pvalb", round(sd7h[sd7h.cell_type == "Pvalb"].CV.astype(float).median(), 3),
      "Sst", round(sd7h[sd7h.cell_type == "Sst"].CV.astype(float).median(), 3))


## fig7h-pvalb-theta-sst-later

> Opto-tagged Pvalb units phase-lock more strongly than Sst units to theta, and Sst units spike
> about 40 to 50 degrees later than Pvalb units in beta, low gamma and high gamma.

In [ ]:
for lfp in "ab":
    p_ = PH[(PH.period == "dg") & (PH.lfp == lfp) & (PH.cap == "first") & (PH.bands == "results")]
    p_ = p_[p_.unit_id.isin(OPTO.index)].merge(OPTO[["group"]], left_on="unit_id", right_index=True)
    lk = p_[p_.p < 0.001]
    th = lk[lk.band == "theta"]
    for col in ("kappa", "R"):
        a1, b1 = th[th.group == "Pvalb"][col], th[th.group == "Sst"][col]
        print(f"LFP {lfp} theta {col}: median Pvalb {a1.median():.3f} (n {len(a1)}), Sst {b1.median():.3f} (n {len(b1)}), "
              f"MW p = {stats.mannwhitneyu(a1, b1, alternative='two-sided').pvalue:.3f}")
    for band in ("beta", "low_gamma", "high_gamma"):
        x = lk[lk.band == band]
        a1, b1 = x[x.group == "Pvalb"].phase, x[x.group == "Sst"].phase
        dm = (circ_mean_deg(b1) - circ_mean_deg(a1) + 180) % 360 - 180
        print(f"LFP {lfp} {band}: mean phase Pvalb {circ_mean_deg(a1):.1f}, Sst {circ_mean_deg(b1):.1f}, Sst - Pvalb {dm:.1f} deg, "
              f"MW p = {stats.mannwhitneyu(a1, b1, alternative='two-sided').pvalue:.3f} (n {len(a1)}, {len(b1)})")

The opto-tagged claims under every reading of the tagged set (isolation c, a or b, baseline 6
or 10 ms, with or without the per-unit exclusions): two-sided tests as above, LFP (a), first
10,000 spikes, Results bands.

In [ ]:
p_all = PH[(PH.period == "dg") & (PH.lfp == "a") & (PH.cap == "first") & (PH.bands == "results")]
rows = []
for iso in "cab":
    for base in ("opto_base6", "opto_base10"):
        for excl in (True, False):
            o = units[units.group.isin(["Pvalb", "Sst"]) & units[f"iso_{iso}"] & tagged(units, base)].copy()
            if excl:
                o = o[unit_ok(o)]
            o["si"] = W.symmetry_index(o.PL_below, o.PL_above)
            a1, b1 = o[o.group == "Pvalb"], o[o.group == "Sst"]
            row = {"iso": iso, "base": base[-2:], "excl": excl, "n Pvalb": len(a1), "n Sst": len(b1)}
            for c in ("TPW", "REP", "rate_spont5", "rate_dg"):
                row[f"{c} p"] = float("%.1e" % stats.mannwhitneyu(a1[c], b1[c], alternative="two-sided").pvalue)
            row["SI mean Pv"], row["SI mean Sst"] = round(a1.si.mean(), 2), round(b1.si.mean(), 2)
            row["SI t p"] = float("%.1e" % stats.ttest_ind(a1.si.dropna(), b1.si.dropna()).pvalue)
            for c in ("cv_dg_a", "cv_dg_b"):
                row[f"{c} Pv>Sst"] = bool(a1[c].median() > b1[c].median())
                row[f"{c} p"] = float("%.1e" % stats.mannwhitneyu(a1[c].dropna(), b1[c].dropna(), alternative="two-sided").pvalue)
            pp = p_all[p_all.unit_id.isin(o.index)].merge(o[["group"]], left_on="unit_id", right_index=True)
            lk = pp[pp.p < 0.001]
            th = lk[lk.band == "theta"]
            row["theta kappa p"] = float("%.1e" % stats.mannwhitneyu(th[th.group == "Pvalb"].kappa, th[th.group == "Sst"].kappa,
                                                                     alternative="two-sided").pvalue)
            for band in ("beta", "low_gamma", "high_gamma"):
                x = lk[lk.band == band]
                row[f"{band} dphase"] = round((circ_mean_deg(x[x.group == "Sst"].phase) - circ_mean_deg(x[x.group == "Pvalb"].phase) + 180) % 360 - 180, 1)
            rows.append(row)
pd.set_option("display.max_columns", 40)
print(pd.DataFrame(rows).to_string(index=False))


In [ ]:
# The main tagged set by session, and the tests with the Pvalb units of the one session that holds most of them.
print(OPTO.assign(si=OPTO.si.round(3))[["group", "session", "TPW", "REP", "si", "cv_dg_a", "cv_dg_b"]]
      .sort_values(["group", "session"]).round(3).to_string())
top = OPTO[OPTO.group == "Pvalb"].session.value_counts().idxmax()
pv1, ss1 = OPTO[(OPTO.group == "Pvalb") & (OPTO.session == top)], OPTO[OPTO.group == "Sst"]
print(f"Pvalb units of session {top} only: {len(pv1)}; Sst units: {len(ss1)}, by session {ss1.session.value_counts().to_dict()}")
for c in ("TPW", "REP", "cv_dg_a", "cv_dg_b"):
    p = stats.mannwhitneyu(pv1[c].dropna(), ss1[c].dropna(), alternative="two-sided").pvalue
    print(f"{c:8s} median Pvalb {pv1[c].median():.3f}, Sst {ss1[c].median():.3f}, MW p = {p:.2e}")
print(f"symmetry index mean Pvalb {pv1.si.mean():.3f}, Sst {ss1.si.mean():.3f}, "
      f"t-test p = {stats.ttest_ind(pv1.si.dropna(), ss1.si.dropna()).pvalue:.2e}")
pp = p_all[p_all.unit_id.isin(pv1.index.union(ss1.index))].merge(OPTO[["group"]], left_on="unit_id", right_index=True)
th = pp[(pp.p < 0.001) & (pp.band == "theta")]
print(f"theta kappa median Pvalb {th[th.group == 'Pvalb'].kappa.median():.3f}, Sst {th[th.group == 'Sst'].kappa.median():.3f}, "
      f"MW p = {stats.mannwhitneyu(th[th.group == 'Pvalb'].kappa, th[th.group == 'Sst'].kappa, alternative='two-sided').pvalue:.2e}")

**Result: partially reproduced.** Pvalb units lock more strongly to theta (median kappa 1.289 against 0.134, p = 0.003). Sst units spike later by 57.1 degrees in low gamma (p = 0.001) and 67.7 in high gamma (p = 0.005), both above 36 to 55 degrees, and by 19.0 degrees in beta (p = 0.30). In the other tagged sets the low-gamma difference falls in 47.6 to 57.1 degrees and the beta difference stays below 23 degrees. On the 9 Pvalb units of session 797828357 alone the theta kappa test gives p = 0.0010.

**Result: reproduced, resting on an assumption.** Pvalb CV is higher within presentations (median 1.714 against 1.025, Mann-Whitney p = 0.0013, 10 and 5 units). Over whole blocks the medians are nearly equal (1.819 against 1.814, p = 0.31). In the readings table, CV within presentations is higher for Pvalb in every tagged set, with p from 0.00011 to 0.053 (only isolation a without exclusions and with the 6 ms baseline is above 0.05; with the 10 ms baseline that set gives 0.0063); over whole blocks no set gives p below 0.05, and Sst is higher under isolation (b) and under isolation (c) with the 6 ms baseline and no exclusions. The outcome rests on the within-presentation reading: over whole blocks, the period whose CV matches the Source Data best for the wild-type units (fig3a-rs1-excitability), the medians are equal, and the Sst median within presentations (1.025) is far from that of Source Data sheet Fig7h (1.624; Pvalb 1.915). 9 of the 10 Pvalb units come from session 797828357; on those 9 alone CV within presentations gives p = 0.0020 and over whole blocks p = 0.24.

**Result: reproduced, resting on an assumption.** The Pvalb mean symmetry index is lower (0.963 against 1.565), t-test p = 0.0034 (Mann-Whitney p = 0.008), close to the Source Data means (0.924 and 1.448). Without the per-unit exclusions the Sst mean rises to 1.43 (isolation c) or 2.47 (isolation a), and the t-test gives 0.025 and 0.079; isolation (b) gives p = 0.79 with the exclusions and 0.25 without. 9 of the 10 Pvalb units come from session 797828357; on those 9 alone the mean is 0.957 against 1.565, t-test p = 0.0052, so the result rests mainly on one Pvalb mouse.

**Result: not reproduced, resting on an assumption.** Pvalb units have lower REP (median 0.209 against 0.302 ms, p = 0.0027) and lower TPW (0.212 against 0.448 ms, p = 0.013, above 0.01), and their rate is not higher under any period reading (p 0.37 to 0.86), so one of the three holds. Without the per-unit exclusions TPW gives p = 0.0012 and REP 0.00016 (isolation c), and with the exclusions and the 10 ms baseline both give p = 0.0020; either would make it partially reproduced. Only the main set, and its isolation (a) twin with the same units, leaves TPW above 0.01. The 6 ms baseline is the window corresponding to the 2 to 8 ms response window (Methods, Detection of opto-tagged neurons: "a corresponding time window immediately preceding stimulus onset"). No tagged set gives a rate difference. On the 9 Pvalb units of session 797828357 alone TPW gives p = 0.019 and REP 0.0040.

## Claims on the models (outside the dataset)

The simulated EAP features of the 33 models are read from model_features.csv of the paper's
code at commit 1e35126 (an input). `cell_type` is 0 for aspiny (AP) and 1 for spiny (SP)
models (Exp_Model_Classifier.ipynb cell 3); `str_label` names the SP clusters RS1 to RS3,
which Source Data sheets Fig5a and Fig5e call SP1 to SP3 for the same cells.

## fig4d-ap-sp-models-tpw-rep

> Simulated EAPs of aspiny (AP) models have significantly lower TPW and REP than those of spiny
> (SP) models, as in vivo FS against RS.

In [ ]:
AP, SP = model[model.cell_type == 0], model[model.cell_type == 1]
print("models: AP", len(AP), "SP", len(SP), "; Cre lines", model.cre_lines.value_counts().to_dict())
print(f"TPW mean AP {AP.TPW.mean():.3f}, SP {SP.TPW.mean():.3f} ms, t-test p = {stats.ttest_ind(AP.TPW, SP.TPW).pvalue:.2e}")
print(f"REP median AP {AP.REP.median():.3f}, SP {SP.REP.median():.3f} ms, Mann-Whitney p = "
      f"{stats.mannwhitneyu(AP.REP, SP.REP, alternative='two-sided').pvalue:.2e}")

**Result: reproduced.** AP models have lower TPW (0.662 against 0.831 ms, t-test p = 2.5e-04) and REP (median 0.461 against 0.598 ms, Mann-Whitney p = 4.9e-04).

## fig4f-model-classifier

> A linear SVM trained on TPW and REP of the models separates AP from SP with 82.5% validation
> accuracy, and applied to in vivo units labels most FS units AP and most RS units SP.

As Exp_Model_Classifier.ipynb cell 7: SP models upsampled to the AP count and z-scored; the
in vivo units (reading c, FS upsampled) z-scored on their own; 100 resamplings of 100 splits.

In [ ]:
SEED_MC = seed_of("fig4f-model-classifier")


def model_exp_runs(seed, n_rep, direction):
    rng = np.random.default_rng(seed)
    mlab = model.cell_type.values
    acc, votes = [], None
    for _ in range(n_rep):
        mi = W.upsample_index(mlab, rng)
        Xm = W.zscore(model[["TPW", "REP"]].values[mi])
        ei = W.upsample_index(lab_wt, rng)
        Xe = W.zscore(wt[["TPW", "REP"]].values[ei])
        if direction == "model":
            a, v, cls = W.classifier(Xm, mlab[mi], Xe, "svm", 100, int(rng.integers(2 ** 31)))
            v = pd.DataFrame(v, columns=cls).groupby(ei).sum().reindex(range(len(wt))).fillna(0).values
        else:
            a, v, cls = W.classifier(Xe, lab_wt[ei], Xm, "svm", 100, int(rng.integers(2 ** 31)))
            v = pd.DataFrame(v, columns=cls).groupby(mi).sum().reindex(range(len(model))).values
        acc.append(a)
        votes = v if votes is None else votes + v
    return float(np.mean(acc)), votes, cls


acc_m, votes_m, cls_m = model_exp_runs(SEED_MC, 100, "model")
pred = np.where(votes_m.sum(1) == 0, "none", np.array(["AP", "SP"])[votes_m.argmax(1)])
tab = pd.crosstab(lab_wt, pred)
print(f"model-based classifier, validation accuracy {100 * acc_m:.2f}% (expected 82.5 +- 8.25)")
print("in vivo units (reading c) by majority label:", tab.to_dict())
print("share of FS labelled AP:", round(tab.loc["FS", "AP"] / tab.loc["FS"].sum(), 3),
      "; share of RS labelled SP:", round(tab.loc["RS", "SP"] / tab.loc["RS"].sum(), 3))
print("five seeds (10 resamplings):", [round(100 * model_exp_runs(SEED_MC + 1000 * k, 10, "model")[0], 2) for k in range(5)])
Xmod = model[["TPW", "REP"]].values.astype(float)
a1, b1 = split_first(Xmod, model.cell_type.values, SEED_MC + 1)
a0, b0 = split_first(Xmod, np.random.default_rng(SEED_MC + 2).permutation(model.cell_type.values), SEED_MC + 3)
print(f"models, split before upsampling (100 splits): accuracy {100 * a1:.2f}%, balanced {100 * b1:.2f}%; "
      f"labels permuted: accuracy {100 * a0:.2f}%, balanced {100 * b0:.2f}%")

**Result: reproduced.** The model-based classifier reaches 83.13% validation accuracy (81.86 to 83.68% under the five seeds); applied to the in vivo units of reading (c) it labels 94.7% of FS units AP and 81.0% of RS units SP. Splitting before upsampling gives 83.22% (balanced 81.60%) against 49.33% with permuted labels.

## fig4g-models-map-to-fs-rs

> The experiment-based classifier labels most AP models FS and most SP models RS.

In [ ]:
SEED_EM = seed_of("fig4g-models-map-to-fs-rs")
acc_e, votes_e, cls_e = model_exp_runs(SEED_EM, 100, "exp")
model["exp_label"] = np.array(cls_e)[votes_e.argmax(1)]
tab = pd.crosstab(np.where(model.cell_type == 0, "AP", "SP"), model.exp_label)
print(f"experiment-based classifier accuracy {100 * acc_e:.2f}%; models by majority label: {tab.to_dict()}")
for k in range(5):
    _, v, c = model_exp_runs(SEED_EM + 1000 * k, 10, "exp")
    lab_k = np.array(c)[v.argmax(1)]
    print(f"seed +{1000 * k} (10 resamplings): AP labelled FS {np.sum((model.cell_type == 0) & (lab_k == 'FS'))} of 18, "
          f"SP labelled RS {np.sum((model.cell_type == 1) & (lab_k == 'RS'))} of 15")

**Result: reproduced.** The experiment-based classifier labels 17 of 18 AP models FS and 11 of 15 SP models RS (16 of 18 under one of the five seeds).

## fig4h-cre-lines-map

> Models of excitatory Cre lines (Scnn1a, Rorb, Nr5a1) are mainly classified RS and models of
> inhibitory lines (Pvalb, Sst) mainly FS.

In [ ]:
print(pd.crosstab(model.cre_lines, model.exp_label).to_string())

**Result: partially reproduced.** Nr5a1 (4 of 5) and Rorb (5 of 6) models are mostly RS and Pvalb (9 of 9) and Sst (8 of 9) mostly FS, but Scnn1a models split 2 and 2; pooled, 11 of 15 excitatory models are RS and 17 of 18 inhibitory models FS.

## fig5a-sp-models-three-clusters

> K-means on the simulated multi-channel features of the 15 SP models gives three clusters
> (SP1 to SP3, with 5, 7 and 3 models), as many as the RS clusters in vivo.

In [ ]:
SEED_SP = seed_of("fig5a-sp-models-three-clusters")
Xsp = W.pre_process(SP[["PL_below", "PL_above"]].values)
wc, fk = W.selection_k(Xsp, 9, seed=SEED_SP)
print(f"SP models: elbow K = {W.find_elbow(wc)}, f(K) minimum K = {int(fk.argmin() + 1)}; f(K) {np.round(fk, 3)}")
lab_sp = W.kmeans_labels(Xsp, 3, SEED_SP)
ref = SP.set_index("cell_list").str_label.str.replace("RS", "SP")
names_sp, tab = W.name_by_overlap(SP.cell_list.values, lab_sp, ref, ["SP1", "SP2", "SP3"])
print("K = 3 clusters:", pd.Series(names_sp).value_counts().sort_index().to_dict(), "; against the paper's labels:", tab.to_dict())
print("paper's labels in model_features.csv:", ref.value_counts().sort_index().to_dict())
for k in range(1, 5):
    l = W.kmeans_labels(Xsp, 3, SEED_SP + 1000 * k)
    print(f"seed +{1000 * k}: sizes {sorted(np.bincount(l).tolist(), reverse=True)}")

**Result: reproduced.** Both methods select K = 3 for the 15 SP models, and the clusters have 5, 7 and 3 models, matching the paper's labels one to one, under all five seeds.

## fig5c-sp-rs-two-way-mapping

> A random forest trained on the SP models' multi-channel features identifies SP1 to SP3 with
> more than 94% accuracy and maps RS1 to SP1, RS2 to SP2 and RS3 to SP3; the
> experiment-based classifier maps SP1 to SP3 back to RS1 to RS3.

The SP clusters of fig5a (named by the paper's labels) and the RS clusters of reading (c);
each side z-scored on its own, classes upsampled; 20 resamplings of 100 splits.

In [ ]:
SEED_RF2 = seed_of("fig5c-sp-rs-two-way-mapping")
rs_t = T["c"][T["c"].onech == "RS"].dropna(subset=["PL_below", "PL_above"])
rs_lab = rs_t.six.values
sp_lab = np.array(names_sp)


def two_way(seed, n_rep):
    rng = np.random.default_rng(seed)
    acc_m2, acc_e2 = [], []
    v_rs = np.zeros((len(rs_t), 3), int)
    v_sp = np.zeros((len(SP), 3), int)
    for _ in range(n_rep):
        mi = W.upsample_index(sp_lab, rng)
        Xm = W.zscore(SP[["PL_below", "PL_above"]].values[mi])
        ei = W.upsample_index(rs_lab, rng)
        Xe = W.zscore(rs_t[["PL_below", "PL_above"]].values[ei])
        a, v, c = W.classifier(Xm, sp_lab[mi], Xe, "rf", 100, int(rng.integers(2 ** 31)))
        acc_m2.append(a)
        v_rs += pd.DataFrame(v, columns=c).groupby(ei).sum().reindex(range(len(rs_t))).fillna(0).values.astype(int)
        a, v, c2 = W.classifier(Xe, rs_lab[ei], Xm, "rf", 100, int(rng.integers(2 ** 31)))
        acc_e2.append(a)
        v_sp += pd.DataFrame(v, columns=c2).groupby(mi).sum().reindex(range(len(SP))).values.astype(int)
    return np.mean(acc_m2), np.mean(acc_e2), v_rs, c, v_sp, c2


am, ae, v_rs, c_sp, v_sp, c_rs = two_way(SEED_RF2, 20)
rs_pred = np.array(c_sp)[v_rs.argmax(1)]
sp_pred = np.array(c_rs)[v_sp.argmax(1)]
print(f"model-based RF accuracy {100 * am:.2f}% ; experiment-based RF accuracy {100 * ae:.2f}%")
print("RS clusters to SP labels:", pd.crosstab(rs_lab, rs_pred).to_dict())
print("SP clusters to RS labels:", pd.crosstab(sp_lab, sp_pred).to_dict())
a1, b1 = split_first(SP[["PL_below", "PL_above"]].values.astype(float), sp_lab, SEED_RF2 + 1, model_name="rf")
a0, b0 = split_first(SP[["PL_below", "PL_above"]].values.astype(float), np.random.default_rng(SEED_RF2 + 2).permutation(sp_lab),
                     SEED_RF2 + 3, model_name="rf")
print(f"SP models, split before upsampling (100 splits): accuracy {100 * a1:.2f}%, balanced {100 * b1:.2f}%; "
      f"labels permuted: accuracy {100 * a0:.2f}%, balanced {100 * b0:.2f}%")
print("five seeds (3 resamplings), model-based accuracy:", [round(100 * two_way(SEED_RF2 + 1000 * k, 3)[0], 2) for k in range(5)])

**Result: partially reproduced.** The model-based random forest reaches 93.48%, just below 94% (93.44 to 94.17% under the five seeds, one above 94%), while both mappings hold: the most frequent SP label of RS1, RS2 and RS3 is SP1, SP2 and SP3, and SP1, SP2 and SP3 map back to RS1 (4 of 5), RS2 (7 of 7) and RS3 (3 of 3). Splitting before upsampling gives 85.25% (balanced 86.33%) against 24.50% with permuted labels, so the code's procedure inflates the accuracy.

## fig6b-pvalb-sst-models-symmetry

> The symmetry index separates Pvalb from Sst models, Pvalb models being more symmetric.

In [ ]:
msi = W.symmetry_index(model.PL_below, model.PL_above)
a_, b_ = msi[model.cre_lines == "Pvalb"], msi[model.cre_lines == "Sst"]
print(f"model symmetry index mean Pvalb {a_.mean():.3f} (n {len(a_)}), Sst {b_.mean():.3f} (n {len(b_)}), "
      f"t-test p = {stats.ttest_ind(a_, b_).pvalue:.2e}")
s6b = sd_table("Fig6b")
print("Source Data Fig6b (z-scored) means: Pvalb", round(s6b[s6b.cre_lines == "Pvalb"]["Symmetry index"].astype(float).mean(), 3),
      "Sst", round(s6b[s6b.cre_lines == "Sst"]["Symmetry index"].astype(float).mean(), 3),
      "; t-test p =", "%.2e" % stats.ttest_ind(s6b[s6b.cre_lines == "Pvalb"]["Symmetry index"].astype(float),
                                             s6b[s6b.cre_lines == "Sst"]["Symmetry index"].astype(float)).pvalue)

**Result: reproduced.** The model symmetry index is lower for Pvalb than Sst (mean 2.692 against 7.739, t-test p = 9.98e-04), as the Source Data gives.

## Claims on the in vitro cells and the models' parameters (data elsewhere)

These claims use the in vitro recordings and reconstructions of the Allen Cell Types Database
and the models' fitted conductances. The models' parameters are not in the paper's model
repository (it holds one example model), so the conductance and morphology claims are tested
on the per-cell values the paper published in its Source Data, which repeats the paper's
statistics on its own numbers. The in vitro features are also read from the Allen Cell Types
Database API (EphysFeature, by specimen id) to check the Source Data values: adaptation,
f-I slope, tau, rheobase (threshold_i_long_square), resting potential (vrest) and rate
(1000 / avg_isi, which equals the Source Data firing_rate). Spike width is not an API
feature, so it is taken from the Source Data only.

Cohen's d uses the pooled SD (a standard choice).

In [ ]:
import json as _json


def api_features(ids):
    url = ("https://api.brain-map.org/api/v2/data/query.json?criteria=model::EphysFeature,"
           "rma::criteria,%5Bspecimen_id$in" + ",".join(str(i) for i in ids) + "%5D&num_rows=200")
    with urllib.request.urlopen(url) as r:
        return pd.DataFrame(_json.load(r)["msg"]).set_index("specimen_id")


def cohen_d(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    sp_ = np.sqrt(((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1)) / (len(a) + len(b) - 2))
    return (a.mean() - b.mean()) / sp_


s4e = sd_table("Fig4e").set_index("specimen__id")
api = api_features(s4e.index.astype(int))
api["rate"] = 1000 / api.avg_isi
print("Allen API cells found:", len(api), "of", len(s4e))
for a_, b_ in [("adaptation", "adaptation"), ("f_i_curve_slope", "f_i_curve_slope"), ("tau", "tau"),
               ("threshold_i_long_square", "rheobase"), ("vrest", "resting_potential"), ("rate", "firing_rate")]:
    d = np.abs(api.loc[s4e.index.astype(int), a_].values - s4e[b_].astype(float).values)
    print(f"   {b_:18s} max |API - Source Data| = {np.nanmax(d):.2e}")
for col, src in [("APwidth", "sd"), ("adaptation", "api"), ("firing_rate", "api"), ("f_i_curve_slope", "api")]:
    if src == "api":
        v = api.loc[s4e.index.astype(int), {"firing_rate": "rate"}.get(col, col)].values
    else:
        v = s4e[col].astype(float).values
    ap_, sp_ = v[s4e.label.values == "AP"], v[s4e.label.values == "SP"]
    n_nan = int(np.isnan(ap_).sum() + np.isnan(sp_).sum())
    ap_, sp_ = ap_[~np.isnan(ap_)], sp_[~np.isnan(sp_)]
    print(f"{col:16s} (cells without a value left out: {n_nan})", end=" ")
    print(f" ({src}) median AP {np.median(ap_):.3f}, SP {np.median(sp_):.3f}; MW p = "
          f"{stats.mannwhitneyu(ap_, sp_, alternative='two-sided').pvalue:.2e}; t-test p = {stats.ttest_ind(ap_, sp_).pvalue:.2e} (n {len(ap_)}, {len(sp_)})")

**Result: partially reproduced.** The Allen API features equal the Source Data values. Spike width (Source Data), spike rate and f-I slope differ with p below 0.001 (1.1e-05, t-test 1.2e-04, 1.2e-06), and adaptation with p = 0.0013, below 0.05 but not 0.001. Not every cell has every feature: spike width is missing for 6 cells, adaptation for 5, rate for 3, and these are left out.

## fig5j-sp1-rate-tau

> In vitro, SP1 neurons reach a higher maximum spike rate than SP2 neurons and are more
> electrotonically compact (differing cellular time constant).

In [ ]:
s5j = sd_table("Fig5j")
s5a = sd_table("Fig5a")
ids5 = s5a.cell_list.astype(int).values
api5 = api.loc[ids5]
lab5 = s5j.str_cell_type.values
for col, v in [("rate (API)", api5.rate.values), ("tau (API)", api5.tau.values),
               ("rate (Source Data)", s5j.firing_rate.astype(float).values), ("tau (Source Data)", s5j.tau.astype(float).values)]:
    a1, a2 = v[lab5 == "SP1"], v[lab5 == "SP2"]
    a1, a2 = a1[~np.isnan(a1)], a2[~np.isnan(a2)]
    print(f"{col}: mean SP1 {a1.mean():.2f}, SP2 {a2.mean():.2f}, Cohen's d SP1-SP2 {cohen_d(a1, a2):.2f} (n {len(a1)}, {len(a2)})")
d5 = np.abs(s5j.firing_rate.astype(float).values - api5.rate.values)
print("Source Data Fig5j rates against API 1000/avg_isi: equal for", int(np.sum(d5 < 1e-6)), "of", len(d5),
      "; API avg_isi missing for", int(np.isnan(api5.rate.values).sum()))

**Result: reproduced.** SP1 against SP2 gives Cohen's d = 0.89 for spike rate (SP1 higher, 14.58 against 11.23 Hz; one SP2 cell has no API rate) and 1.13 for tau.

## fig6g-pvalb-sst-in-vitro

> In vitro, Pvalb and Sst neurons differ in maximum spike rate, rheobase and resting potential
> among other properties.

In [ ]:
s6g = sd_table("Fig6g").set_index("specimen__id")
api6 = api_features(s6g.index.astype(int))
api6["rate"] = 1000 / api6.avg_isi
cre6 = s6g.cre_lines.values
print("API avg_isi missing for", int(api6.rate.isna().sum()), "of", len(api6), "cells; Source Data firing_rate shown as well")
for col in ("rate", "threshold_i_long_square", "vrest", "sd firing_rate"):
    v = (s6g.firing_rate.astype(float).values if col == "sd firing_rate"
         else api6.loc[s6g.index.astype(int), col].values)
    a1, a2 = v[cre6 == "Pvalb"], v[cre6 == "Sst"]
    a1, a2 = a1[~np.isnan(a1)], a2[~np.isnan(a2)]
    print(f"{col:24s} median Pvalb {np.median(a1):.2f}, Sst {np.median(a2):.2f}, MW p = {stats.mannwhitneyu(a1, a2, alternative='two-sided').pvalue:.4f}, "
          f"t-test p = {stats.ttest_ind(a1, a2).pvalue:.4f} (n {len(a1)}, {len(a2)})")

**Result: reproduced.** From the Allen API features, maximum spike rate (Mann-Whitney p = 0.0011), rheobase (0.0008) and resting potential (0.0081) differ between Pvalb and Sst cells. The paper gives p = 0.0067 for the rate; here the rate gives 0.0011 by either source, and 0.0068 is the t-test p of the resting potential, so the paper's rate p may belong to another feature or test.

## fig5e-ca-lva

> Axonal low-voltage-activated Ca conductance is higher in SP1 and SP3 models than in SP2
> models, with a large effect size.

In [ ]:
s5e = sd_table("Fig5e")
g = s5e.gbar_Ca_LVA.astype(float).values
l5 = s5e.str_cell_type.values
for a in ("SP1", "SP3"):
    print(f"{a} against SP2: mean {g[l5 == a].mean():.5f} against {g[l5 == 'SP2'].mean():.5f}, Cohen's d {cohen_d(g[l5 == a], g[l5 == 'SP2']):.2f}")

**Result: reproduced** on the Source Data values. Axonal Ca LVA is higher in SP1 and SP3 than SP2 (Cohen's d 2.07 and 2.12).

## fig6cf-kv31

> Pvalb and Sst models differ significantly in three conductances, the largest effect being
> dendritic Kv3.1.

Source Data sheet Fig6cf lists five conductances, not all of the models' parameters.

In [ ]:
s6c = sd_table("Fig6cf")
for col in s6c.columns[4:]:
    v = s6c[col].astype(float).values
    a1, a2 = v[s6c.cre_lines.values == "Pvalb"], v[s6c.cre_lines.values == "Sst"]
    print(f"{col:18s} MW p = {stats.mannwhitneyu(a1, a2, alternative='two-sided').pvalue:.4f}, |d| = {abs(cohen_d(a1, a2)):.2f}")

**Result: partially reproduced** on the Source Data values. Of the five conductances the sheet lists, three differ (dendritic Kv3.1 p = 0.0217, dendritic NaV 0.0423, axonal e_pas 0.0341), and dendritic Kv3.1 has the largest effect (|d| = 1.29). The criterion tests every conductance of the models, and only these five are public, so "exactly three" cannot be checked.

## fig5g-bifurcation-velocity-sp and fig6e-bifurcation-velocity-inhibitory

> Across SP models the morphological bifurcation distance is linearly related to the inverse
> spike propagation velocity (slope 2.7, r = 0.8).

> Across Pvalb and Sst models the bifurcation distance above and below the soma is linearly
> related to the inverse EAP propagation velocity (slope 2.67, r = 0.8).

Regression of 1/V on bifurcation distance, above and below pooled (each model gives two
points), on the Source Data values; the direction (1/V on distance, or the reverse) is not
stated, so both are shown.

In [ ]:
def bif_reg(name):
    t = sd_table(name)
    x = np.r_[t["Bifurcation distance above soma"].astype(float), t["Bifurcation distance below soma"].astype(float)]
    y = np.r_[t["1/V_above"].astype(float), t["1/V_below"].astype(float)]
    a = stats.linregress(x, y)
    b = stats.linregress(y, x)
    print(f"{name}: n {len(x)} points; 1/V on distance slope {a.slope:.2f}, r {a.rvalue:.2f}, p {a.pvalue:.1e}; "
          f"distance on 1/V slope {b.slope:.3f}")
bif_reg("Fig5g")

**Result: reproduced** on the Source Data values: 1/V on bifurcation distance gives slope 2.71, r = 0.80, p = 1.0e-07.

In [ ]:
bif_reg("Fig6d")

**Result: reproduced** on the Source Data values: slope 2.68, r = 0.80, p = 5.1e-09.

## fig6d-sst-asymmetric-morphology

> Sst morphologies are asymmetric, with bifurcation distance above the soma longer than below
> (above against below, p = 0.03), while Pvalb morphologies are symmetric (above against below,
> p = 0.45).

In [ ]:
t6 = sd_table("Fig6d")
for g_ in ("Pvalb", "Sst"):
    x = t6[t6.label == g_]
    up, dn = x["Bifurcation distance above soma"].astype(float).abs(), x["Bifurcation distance below soma"].astype(float).abs()
    print(f"{g_}: mean |above| {up.mean():.2f}, |below| {dn.mean():.2f}, t-test p = {stats.ttest_ind(up, dn).pvalue:.3f}")

**Result: reproduced** on the Source Data values: Sst |above| is larger than |below| (4.04 against 1.70, p = 0.030), Pvalb shows no difference (p = 0.455).

## figs8-rotation-symmetry (blocked)

> The angle between the probe and the morphology changes multi-channel EAP metrics, including
> the symmetry index, of Pvalb and Sst models, with some angles enlarging and some hiding the
> Pvalb against Sst difference.

Not run: it needs simulations of the 18 inhibitory models at several angles, and neither the
simulations nor the models' parameters are public. The paper's model repository holds the
set-up of one spiny model only (cell 395830185), and the dandiset holds no simulations.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| source-data-units-link | partially reproduced (13 of 25 sessions; all 566 of their Source Data units found in VISp) |
| fig1d-unit-count | not reproduced (57.0 or 140.0 units per mouse against 48) |
| fig2-one-channel-two-clusters | reproduced |
| fig2-fs-rs-counts | reproduced (assumed; FS 25.4% under reading c) |
| figs3b-fs-faster | not reproduced (FS faster, but no reading puts both medians within tolerance) |
| fig2-rs-more-asymmetric | reproduced |
| fig2d-three-subclusters-each | not reproduced (FS gives K = 2; the paper's features give 3) |
| figs4-rs1-faster-above | reproduced |
| figs4-rs-spread | reproduced |
| figs4-fs-spread-absent | reproduced (assumed) |
| figs5-depth-distribution | not reproduced (assumed; p = 0.044 with the CSD centre here, 0.36 on the Source Data labels; partially reproduced proposed, the round-3 verifier would give reproduced) |
| fig6a-fs1-symmetric | reproduced (assumed) |
| fig4g-experiment-classifier-fs-rs | reproduced |
| fig5d-experiment-classifier-rs | partially reproduced (98.6%) |
| fig3a-isi-fs-rs | partially reproduced (assumed; both medians within 10%) |
| fig3a-rs1-excitability | partially reproduced (assumed; one measure of three) |
| fig3a-rs23-visual-responses | not reproduced (assumed) |
| fig3a-fs-visual-absent | reproduced (assumed) |
| fig3c-fs-more-phase-locked | not reproduced (assumed; 2 bands significant) |
| fig3c-rs-prefers-theta | reproduced |
| figs7-spontaneous-less-locked | partially reproduced (direction only) |
| fig3c-fs-stronger-earlier | not reproduced |
| fig3d-rs2-gamma-locking | not reproduced |
| fig3e-fs-cluster-locking | partially reproduced (assumed) |
| figs6c-fs-stronger-across-depth | not reproduced (assumed) |
| figs6d-rs3-supragranular-beta | not reproduced (assumed) |
| figs6c-granular-phase | not reproduced (assumed) |
| fig7-opto-unit-counts | not reproduced |
| fig7d-pvalb-fs-sst-diffuse | reproduced |
| fig7e-pvalb-sst-one-channel | not reproduced (assumed; REP only at p < 0.01) |
| fig7g-pvalb-more-symmetric | reproduced (assumed) |
| fig7h-pvalb-higher-cv | reproduced (assumed) |
| fig7h-pvalb-theta-sst-later | partially reproduced |
| fig4d, fig4f, fig4g-models, fig5a, fig6b | reproduced (model features) |
| fig4h-cre-lines-map, fig5c-sp-rs-two-way-mapping | partially reproduced (model features) |
| fig4e-in-vitro-ap-sp | partially reproduced (adaptation p = 0.0013) |
| fig5j, fig6g, fig5e, fig5g, fig6d, fig6e | reproduced (Source Data and Allen API values) |
| fig6cf-kv31 | partially reproduced (5 of the models' conductances are public) |
| figs8-rotation-symmetry | blocked (not runnable: the models and simulations are not public) |

**How the analysis differs from the paper's.** Half of the paper's wild-type sessions and
about half of its opto-tagging mice are not in the dandiset. The isolation step behind the
code's "good" flag is not stated. Neither reading of the criterion gives the paper's units;
the claims after fig1d-unit-count use reading (c), the AllenSDK filter with at least 100 spikes
in the first 5-min gray screen, which keeps every reading (a) unit of the code's tables and few
others (an assumption). The code's 3 SD exclusions are taken over the values present, as NaN
in 1/V above would otherwise switch them off. FS is clustered with K forced to 3 for the claims
on FS1 to FS3. Depth is taken from the layer 4 centre computed here from the flash CSD, the method the
criterion of figs5-depth-distribution names; the code's tables give one centre per probe (2740 um
on 8 of the 13 probes), which differs from it by -420 to +340 um and is shown as a variant.
Neither centre is validated. 9 of the 10 Pvalb units of the main tagged set come from one
session. The opto-tagged units pass the code's
per-unit exclusions. With about half the paper's units, several weaker effects are not
significant here; the outcomes under every reading are printed in each section.

**What was found about the dataset.** The 13 held sessions contain every one of their 566
Source Data units, in VISp, with the Source Data's waveform features recomputed exactly for
most units; the drifting-grating metrics and the code's firing rates are reproduced exactly
from the files. Mean waveforms are labelled volts but hold microvolts. The electrodes tables'
CCF coordinates hold NaN in 6 of the 24 sessions (every value in 3 of them), and in the other
18 y equals z on every channel. The paper's other 12 wild-type sessions are not among the
dandiset's 32 session files.